# Sparse Control Structures in DiTs: Cross-Model Reproduction
**Models:** FLUX.1-schnell · FLUX.1-dev · PixArt-Sigma-XL-2-1024-MS
**Experiments** (each answers one open question):

| Exp | Question | Method |
|---|---|---|
| E1 | Register channel identity + channel state in ordinary vs register tokens; depth window | census: per-layer norms, register vectors, per-channel profile |
| E2 | Core replication: direction vs magnitude causes sink selection | boundary ablation (sham / clamp / rotate) + light [H,N] sink capture |
| E3 | Why does high norm protect direction | measured update magnitudes vs exact small-angle drift prediction |
| E4 | Why attention prefers the register channel | analytic e_ch → LN → W_k → RMSNorm vs captured query consensus + random-channel null |
| E5 | Which MLP neurons write the register channel | fc2 row-energy ranking + activation capture + targeted neuron ablation |
| E6 | Text tokens (Flux only) | text-stream census + image→padding attention masking |

**Reproducibility policy:** pinned package versions (cell 1), fixed global seed, deterministic RNG via crc32 (survives session restarts), per-model output folders on Drive, every run resume-safe (skips files that exist), config stamped to `config.json`.

**Memory policy:** attention is never materialised as [H,N,N]; the `LightSinkProcessor` accumulates head-mean incoming mass in query chunks, so 1024×1024 and Dev's 28-step schedule are safe.

**Register channel is discovered per model, not assumed:** E1 finds each model's dominant register channel empirically (154 is a FLUX fact; PixArt's may differ, its fixed bottom-right register vs FLUX's noise-picked winners is a measured contrast, not an assumption).

**Validation status:** the chunked sink capture, drift prediction, analytic-QK path, neuron-ablation weight restore, and padding mask were unit-tested against exact references on CPU. Model-dependent wiring (hook points, block structures) matches diffusers 0.35.1 and should be sanity-checked on first GPU run (E1's census figure is the canary: the FLUX outlier window must reproduce layers ~18 to 40).

In [ ]:
import os

print(os.listdir("/content/drive"))
print(os.path.exists("/content/drive/MyDrive"))

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path("/content/drive/MyDrive/sparse_structures_repro")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# [CELL 1] ------------------------------------------------------------
# Setup: pinned environment, seeds, paths, config
# -------------------------------------------------------------------------
# Sparse structures in DiTs, cross-model reproduction notebook
# Models: FLUX.1-schnell, FLUX.1-dev, PixArt-Sigma-XL-2-1024-MS
# Experiments:
#   E1  Register census + register-channel identity (descriptive)
#   E2  Direction-vs-magnitude causal ablation (core replication)
#   E3  Inertia: predicted vs observed angular drift
#   E4  Analytic QK alignment of the register channel
#   E5  Register-neuron search + ablation in the birth-block MLP
#   E6  Text-stream census + padding-sink masking (Flux only)
# Reproducibility: pinned versions, fixed seeds, resume-safe, config stamps.

!pip install -q diffusers==0.35.1 transformers==4.56.2 accelerate==0.34.2 \
     sentencepiece protobuf pandas==2.2.2 matplotlib seaborn lpips

from __future__ import annotations
import json, math, os, zlib
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")
torch.backends.cuda.matmul.allow_tf32 = True

GLOBAL_SEED = 0
def seed_everything(seed: int = GLOBAL_SEED) -> None:
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything()



PROMPTS = [
    "A close-up portrait of an astronaut in a reflective helmet, cinematic lighting, ultra detailed",
    "A futuristic city street at night after rain, neon reflections, detailed architecture",
    "A red fox sitting in a snowy forest, shallow depth of field, cinematic",
    "A single white teacup on a plain background, minimal, soft light",
    "A dense forest canopy seen from below, sunbeams, high detail",
]
SEEDS = [0, 42, 1234]


@dataclass
class RunConfig:
    model_key: str                      # "flux-schnell" | "flux-dev" | "pixart-sigma"
    height: int = 1024
    width: int = 1024
    prompts: List[str] = field(default_factory=lambda: list(PROMPTS))
    seeds: List[int] = field(default_factory=lambda: list(SEEDS))
    seed_ratio: float = 3.0             # register = norm > seed_ratio x median
    k_max: int = 8
    capture_steps: Optional[List[int]] = None   # None = model default subset
    dtype: str = "bfloat16"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    @property
    def out(self) -> Path:
        p = DRIVE_ROOT / self.model_key
        p.mkdir(parents=True, exist_ok=True)
        return p

    def stamp(self) -> None:
        (self.out / "config.json").write_text(json.dumps(asdict(self), indent=2))


def det_rng(*tags) -> torch.Generator:
    """Deterministic across sessions (crc32, not python hash)."""
    h = zlib.crc32("|".join(map(str, tags)).encode())
    return torch.Generator(device="cpu").manual_seed(int(h))




In [ ]:
# [CELL 2] ------------------------------------------------------------
# Model adapters: one interface, three models
# -------------------------------------------------------------------------
# Adapter contract:
#   load()                          -> sets .pipe, .transformer
#   iter_blocks()                   -> [(layer_idx, name, module, kind)] kind in {dual,single,basic}
#   boundary_block()                -> module whose pre-hook input is the image stream
#   image_slice(seq_len)            -> (start, end) of image tokens inside self-attn sequence
#   n_image_tokens()                -> tokens at current resolution
#   default_steps / guidance
#   attach_light_sink_capture(...)  -> chunked [H, N] incoming-mass capture (no NxN materialised)
#   run(prompt, seed, hooks_ctx)    -> generates one image
# Notes:
#   * Flux single blocks: self-attn sequence = [text | image]; block pre-hook
#     hidden_states = image-only (verified on diffusers 0.35.1).
#   * PixArt attn1 sequence = image-only; text enters via cross-attn (attn2) only,
#     so E6 (text-in-residual experiments) is Flux-only by architecture.

class LightSinkProcessor:
    """Attention processor that computes the block output with SDPA but ALSO
    accumulates head-mean incoming attention over image queries, chunked so the
    full [H, N, N] matrix is never materialised. Output: sink dict tag ->
    {"incoming": [H, K_img], "text_incoming": [H, K_text] or None}."""

    def __init__(self, sink: dict, tag: str, image_slice_fn, chunk: int = 256):
        self.sink, self.tag, self.image_slice_fn, self.chunk = sink, tag, image_slice_fn, chunk

    def __call__(self, attn, hidden_states, encoder_hidden_states=None,
                 attention_mask=None, image_rotary_emb=None, **kw):
        x = hidden_states
        ctx = encoder_hidden_states if encoder_hidden_states is not None else hidden_states
        q = attn.to_q(x); k = attn.to_k(ctx); v = attn.to_v(ctx)
        B, Sq, _ = q.shape
        Sk = k.shape[1]
        H = attn.heads; Dh = q.shape[-1] // H
        q = q.view(B, Sq, H, Dh); k = k.view(B, Sk, H, Dh); v = v.view(B, Sk, H, Dh)
        if getattr(attn, "norm_q", None) is not None: q = attn.norm_q(q)
        if getattr(attn, "norm_k", None) is not None: k = attn.norm_k(k)
        if image_rotary_emb is not None:
            from diffusers.models.embeddings import apply_rotary_emb
            q = apply_rotary_emb(q, image_rotary_emb, sequence_dim=1)
            k = apply_rotary_emb(k, image_rotary_emb, sequence_dim=1)
        qh = q.permute(0, 2, 1, 3); kh = k.permute(0, 2, 1, 3); vh = v.permute(0, 2, 1, 3)

        i0, i1 = self.image_slice_fn(Sq)
        scale = Dh ** -0.5
        inc = torch.zeros(H, Sk, device=q.device, dtype=torch.float32)
        nq = 0
        for s in range(i0, i1, self.chunk):
            e = min(s + self.chunk, i1)
            logits = torch.einsum("bhqd,bhkd->bhqk", qh[:, :, s:e].float(), kh.float()) * scale
            probs = logits.softmax(dim=-1)
            inc += probs[0].sum(dim=1)          # [H, Sk]
            nq += (e - s)
        inc /= max(nq, 1)
        j0, j1 = self.image_slice_fn(Sk)
        self.sink[self.tag] = {
            "incoming": inc[:, j0:j1].cpu(),
            "text_incoming": inc[:, :j0].cpu() if j0 > 0 else None,
        }

        out = F.scaled_dot_product_attention(qh, kh, vh)
        out = out.permute(0, 2, 1, 3).reshape(B, Sq, H * Dh).to(hidden_states.dtype)
        if getattr(attn, "to_out", None) is not None and encoder_hidden_states is not None:
            out = attn.to_out[0](out); out = attn.to_out[1](out)
        return out


class FluxAdapter:
    KIND = "flux"
    def __init__(self, cfg: RunConfig):
        self.cfg = cfg
        self.model_id = {"flux-schnell": "black-forest-labs/FLUX.1-schnell",
                         "flux-dev": "black-forest-labs/FLUX.1-dev"}[cfg.model_key]
        self.default_steps = 4 if cfg.model_key == "flux-schnell" else 28
        self.guidance = 0.0 if cfg.model_key == "flux-schnell" else 3.5
        self.text_length = 512

    def load(self):
        from diffusers import FluxPipeline
        dt = {"bfloat16": torch.bfloat16, "float16": torch.float16}[self.cfg.dtype]
        self.pipe = FluxPipeline.from_pretrained(self.model_id, torch_dtype=dt).to(self.cfg.device)
        self.pipe.set_progress_bar_config(disable=True)
        self.transformer = self.pipe.transformer
        return self

    def n_image_tokens(self):
        return (self.cfg.height // 16) * (self.cfg.width // 16)

    def iter_blocks(self):
        out = []
        for i, b in enumerate(self.transformer.transformer_blocks):
            out.append((i, f"dual_{i}", b, "dual"))
        for i, b in enumerate(self.transformer.single_transformer_blocks):
            out.append((19 + i, f"single_{i}", b, "single"))
        return out

    def boundary_block(self):
        return self.transformer.single_transformer_blocks[0]

    def image_slice(self, seq_len):
        return (self.text_length, seq_len) if seq_len > self.n_image_tokens() else (0, seq_len)

    def attach_light_sink_capture(self, sink: dict, layer_names: Sequence[str]):
        originals = {}
        for idx, name, block, kind in self.iter_blocks():
            if name in layer_names:
                originals[name] = block.attn.processor
                block.attn.set_processor(LightSinkProcessor(sink, name, self.image_slice))
        return originals

    def restore_processors(self, originals):
        for idx, name, block, kind in self.iter_blocks():
            if name in originals:
                block.attn.set_processor(originals[name])

    def run(self, prompt: str, seed: int, steps: Optional[int] = None):
        gen = torch.Generator(device=self.cfg.device).manual_seed(seed)
        return self.pipe(prompt=prompt, height=self.cfg.height, width=self.cfg.width,
                         num_inference_steps=steps or self.default_steps,
                         guidance_scale=self.guidance, generator=gen,
                         max_sequence_length=self.text_length, output_type="pil").images[0]


class PixArtAdapter:
    KIND = "pixart"
    def __init__(self, cfg: RunConfig):
        self.cfg = cfg
        self.model_id = "PixArt-alpha/PixArt-Sigma-XL-2-1024-MS"
        self.default_steps = 20
        self.guidance = 4.5
        self.text_length = 0            # text is cross-attn only; not in self-attn stream

    def load(self):
        from diffusers import PixArtSigmaPipeline
        dt = {"bfloat16": torch.bfloat16, "float16": torch.float16}[self.cfg.dtype]
        self.pipe = PixArtSigmaPipeline.from_pretrained(self.model_id, torch_dtype=dt).to(self.cfg.device)
        self.pipe.set_progress_bar_config(disable=True)
        self.transformer = self.pipe.transformer
        return self

    def n_image_tokens(self):
        # PixArt-Sigma: VAE /8, patch_size 2 -> /16 total
        return (self.cfg.height // 16) * (self.cfg.width // 16)

    def iter_blocks(self):
        return [(i, f"basic_{i}", b, "basic")
                for i, b in enumerate(self.transformer.transformer_blocks)]

    def boundary_block(self):
        # Known PixArt high-norms sit in layers ~13-18, so the boundary is set after 13
        return self.transformer.transformer_blocks[14]

    def image_slice(self, seq_len):
        return (0, seq_len)

    def attach_light_sink_capture(self, sink: dict, layer_names: Sequence[str]):
        originals = {}
        for idx, name, block, kind in self.iter_blocks():
            if name in layer_names:
                originals[name] = block.attn1.processor
                block.attn1.set_processor(LightSinkProcessor(sink, name, self.image_slice))
        return originals

    def restore_processors(self, originals):
        for idx, name, block, kind in self.iter_blocks():
            if name in originals:
                block.attn1.set_processor(originals[name])

    def run(self, prompt: str, seed: int, steps: Optional[int] = None):
        gen = torch.Generator(device=self.cfg.device).manual_seed(seed)
        return self.pipe(prompt=prompt, height=self.cfg.height, width=self.cfg.width,
                         num_inference_steps=steps or self.default_steps,
                         guidance_scale=self.guidance, generator=gen,
                         output_type="pil").images[0]


def make_adapter(cfg: RunConfig):
    return (FluxAdapter(cfg) if cfg.model_key.startswith("flux") else PixArtAdapter(cfg)).load()


def block_post_image(out):
    """Image hidden states from a block output (tuple for Flux, tensor for PixArt)."""
    return out[1] if isinstance(out, tuple) and len(out) >= 2 else out


In [ ]:
# [CELL 3] ------------------------------------------------------------
# E1: Register census + register-channel identity (descriptive)
# -------------------------------------------------------------------------
# Per (prompt, seed, captured step, layer): per-token norms; register set
# (norm > seed_ratio x median); register full vectors at focus layers;
# per-channel token-mean-abs. Then:
#   * per-model REGISTER CHANNEL = argmax energy of the mean register unit vector
#   * x_ch distribution: registers vs ordinary tokens (the channel-state question)
#   * birth/dissolution trajectory across depth (descriptive)
#   * winner grid positions (fixed for PixArt vs noise for Flux)

def run_census(cfg: RunConfig, adapter, focus_every: int = 4) -> None:
    out = cfg.out / "census"; out.mkdir(exist_ok=True)
    blocks = adapter.iter_blocks()
    focus = {name for i, (idx, name, b, k) in enumerate(blocks) if idx % focus_every == 0}
    steps_all = adapter.default_steps
    cap_steps = cfg.capture_steps or sorted({0, steps_all // 2, steps_all - 1})
    state = {"step": -1, "rows": [], "vecs": []}

    def make_post(idx, name):
        def post(mod, args, kwargs, o):
            if state["step"] not in cap_steps:
                return
            h = block_post_image(o)[0].float()                     # [N, C]
            norms = h.norm(dim=-1); med = norms.median()
            reg = torch.nonzero(norms > cfg.seed_ratio * med).flatten()
            if reg.numel() > cfg.k_max:
                reg = reg[torch.argsort(norms[reg], descending=True)[: cfg.k_max]]
            state["rows"].append(dict(layer=idx, name=name, step=state["step"],
                                      median_norm=float(med), max_norm=float(norms.max()),
                                      n_registers=int(reg.numel()),
                                      register_ids=json.dumps(reg.tolist())))
            if name in focus and reg.numel() > 0:
                state["vecs"].append(dict(layer=idx, step=state["step"],
                                          ids=reg.cpu(), vecs=h[reg].cpu(),
                                          ch_mean_abs=h.abs().mean(0).cpu()))
        return post

    handles = [adapter.transformer.register_forward_pre_hook(
        lambda m, a, k: state.__setitem__("step", state["step"] + 1), with_kwargs=True)]
    for idx, name, block, kind in blocks:
        handles.append(block.register_forward_hook(make_post(idx, name), with_kwargs=True))
    try:
        for pid, prompt in enumerate(cfg.prompts):
            for seed in cfg.seeds:
                fp = out / f"census_p{pid}_s{seed}.pt"
                if fp.exists():
                    continue
                state["step"], state["rows"], state["vecs"] = -1, [], []
                adapter.run(prompt, seed)
                torch.save({"prompt_id": pid, "seed": seed,
                            "rows": state["rows"], "vecs": state["vecs"]}, fp)
                print("saved", fp.name)
    finally:
        for h in handles: h.remove()
        torch.cuda.empty_cache()


def analyze_census(cfg: RunConfig) -> dict:
    out = cfg.out / "census"
    caps = [torch.load(f, weights_only=False) for f in sorted(out.glob("census_p*.pt"))]

    rows = pd.DataFrame([r | {"prompt_id": c["prompt_id"], "seed": c["seed"]}
                         for c in caps for r in c["rows"]])
    rows["ratio"] = rows["max_norm"] / rows["median_norm"]
    traj = rows.groupby("layer")["ratio"].mean()

    # register channel: mean sign-aligned unit vector over all captured register vecs
    U, x_reg, ch_prof = [], [], []
    for c in caps:
        for v in c["vecs"]:
            for i in range(v["vecs"].shape[0]):
                u = v["vecs"][i]; U.append(u / u.norm().clamp(min=1e-6))
            ch_prof.append(v["ch_mean_abs"])
    result = {"register_channel": None}
    if U:
        U = torch.stack(U)
        Ua = U * torch.sign(U @ U[0]).unsqueeze(-1)
        vstar = Ua.mean(0); vstar = vstar / vstar.norm()
        e = vstar.pow(2); order = torch.argsort(e, descending=True)
        ch = int(order[0])
        result.update(register_channel=ch, second_channel=int(order[1]),
                      top1_energy=float(e[order[0]]), vstar=vstar,
                      mean_cos=float((Ua @ vstar).mean()))
        # x_ch in registers vs ordinary tokens (needs raw h; use stored reg vecs vs implied floor)
        reg_vals = torch.stack([abs(v["vecs"][i][ch]) for c in caps for v in c["vecs"]
                                for i in range(v["vecs"].shape[0])])
        floor = torch.stack(ch_prof).mean(0)[ch]
        result.update(reg_x_ch_mean=float(reg_vals.mean()), floor_x_ch=float(floor),
                      spike_to_floor=float(reg_vals.mean() / max(float(floor), 1e-6)))
    torch.save(result | {"trajectory": traj}, out / "census_summary.pt")

    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].semilogy(traj.index, traj.values, marker=".")
    ax[0].axhline(cfg.seed_ratio, ls="--", color="gray")
    ax[0].set_xlabel("layer"); ax[0].set_ylabel("max/median norm")
    ax[0].set_title(f"{cfg.model_key}: outlier ratio by depth (register window where above dashed)")
    if U is not None and len(U):
        ax[1].semilogy(e[order][:100].numpy(), marker=".")
        ax[1].set_title(f"register-direction energy by channel rank (top-1 = ch {result['register_channel']}, "
                        f"{result['top1_energy']:.2f})")
    fig.tight_layout(); fig.savefig(out / "fig_census.png", dpi=180); plt.show()
    print({k: v for k, v in result.items() if k not in ("vstar", "trajectory")})
    return result




In [ ]:
# [CELL 4] ------------------------------------------------------------
# E2: Direction vs magnitude causal ablation (core replication)
# -------------------------------------------------------------------------
# Conditions at the model's boundary block, every denoising step:
#   sham        select registers, change nothing (instrumentation anchor)
#   seed_clamp  rescale register norms to median (direction kept)
#   seed_rotate replace register directions with random units (magnitude kept)
# Metric: fraction of heads whose top-1 sink token is unchanged vs sham, from
# light [H, N] captures at a small set of observation layers.

E2_CONDITIONS = ["sham", "seed_clamp", "seed_rotate"]

class BoundaryHook:
    def __init__(self, cfg: RunConfig):
        self.cfg = cfg; self.condition = "sham"
        self.pid = 0; self.seed = 0; self.step = -1
        self.log: List[dict] = []

    def __call__(self, module, args, kwargs):
        in_kw = isinstance(kwargs, dict) and "hidden_states" in kwargs
        x = kwargs["hidden_states"] if in_kw else (args[0] if args else None)
        if x is None: return None
        norms = x[0].float().norm(dim=-1); med = norms.median()
        reg = torch.nonzero(norms > self.cfg.seed_ratio * med).flatten()
        if reg.numel() > self.cfg.k_max:
            reg = reg[torch.argsort(norms[reg], descending=True)[: self.cfg.k_max]]
        self.log.append(dict(condition=self.condition, prompt_id=self.pid, seed=self.seed,
                             step=self.step, ids=json.dumps(reg.tolist()), median=float(med)))
        if self.condition == "sham" or reg.numel() == 0:
            return None
        x = x.clone()
        if self.condition == "seed_clamp":
            sc = (med / norms[reg].clamp(min=1e-6)).to(x.dtype)
            x[0, reg] = x[0, reg] * sc.unsqueeze(-1)
        elif self.condition == "seed_rotate":
            g = det_rng("rot", self.pid, self.seed, self.step)
            v = torch.randn(reg.numel(), x.shape[-1], generator=g)
            v = (v / v.norm(dim=-1, keepdim=True)).to(x.device, torch.float32)
            x[0, reg] = (v * norms[reg].unsqueeze(-1)).to(x.dtype)
        if in_kw:
            kwargs = dict(kwargs); kwargs["hidden_states"] = x
            return (args, kwargs)
        return ((x,) + tuple(args[1:]), kwargs)


def run_causal(cfg: RunConfig, adapter, obs_layers: Optional[List[str]] = None) -> None:
    out = cfg.out / "causal"; out.mkdir(exist_ok=True)
    if obs_layers is None:
        names = [n for _, n, _, k in adapter.iter_blocks()]
        if adapter.KIND == "flux":
            obs_layers = ["single_0", "single_4", "single_8"]
        else:
            b = adapter.boundary_block()
            bi = [i for i, (_, n, m, _) in enumerate(adapter.iter_blocks()) if m is b][0]
            obs_layers = [names[bi], names[min(bi + 4, len(names) - 1)], names[min(bi + 8, len(names) - 1)]]
    hook = BoundaryHook(cfg)
    steps_all = adapter.default_steps
    cap_steps = cfg.capture_steps or sorted({0, steps_all // 2, steps_all - 1})

    for cond in E2_CONDITIONS:
        rows_fp = out / f"sinks_{cond}.pt"
        if rows_fp.exists():
            print("skip", cond); continue
        hook.condition = cond
        sink_store, all_caps = {}, []
        step_state = {"i": -1}

        def tf_pre(m, a, k):
            step_state["i"] += 1; hook.step = step_state["i"]
            return None
        h_tf = adapter.transformer.register_forward_pre_hook(tf_pre, with_kwargs=True)
        h_b = adapter.boundary_block().register_forward_pre_hook(hook, with_kwargs=True)
        originals = adapter.attach_light_sink_capture(sink_store, obs_layers)
        img_dir = out / "images" / cond; img_dir.mkdir(parents=True, exist_ok=True)
        try:
            for pid, prompt in enumerate(cfg.prompts):
                for seed in cfg.seeds:
                    hook.pid, hook.seed = pid, seed
                    step_state["i"] = -1
                    per_step = {}
                    def snap_pre(m, a, k):     # snapshot sink_store at captured steps
                        if step_state["i"] in cap_steps and sink_store:
                            per_step[step_state["i"] - 1] = {t: {kk: vv.clone() if vv is not None else None
                                                                 for kk, vv in d.items()}
                                                             for t, d in sink_store.items()}
                        return None
                    img = adapter.run(prompt, seed)
                    img.save(img_dir / f"p{pid}_s{seed}.png")
                    # final-state capture (sink_store holds last step)
                    all_caps.append(dict(prompt_id=pid, seed=seed, step=steps_all - 1,
                                         sinks={t: d["incoming"].argmax(dim=-1) for t, d in sink_store.items()},
                                         text_frac={t: (float(d["text_incoming"].sum(-1).mean())
                                                        if d["text_incoming"] is not None else None)
                                                    for t, d in sink_store.items()}))
        finally:
            h_tf.remove(); h_b.remove(); adapter.restore_processors(originals)
            torch.cuda.empty_cache()
        torch.save(all_caps, rows_fp)
        pd.DataFrame(hook.log).to_csv(out / "boundary_log.csv", index=False)
        print("saved", cond)


def analyze_causal(cfg: RunConfig) -> pd.DataFrame:
    out = cfg.out / "causal"
    data = {c: torch.load(out / f"sinks_{c}.pt", weights_only=False) for c in E2_CONDITIONS
            if (out / f"sinks_{c}.pt").exists()}
    base = {(r["prompt_id"], r["seed"]): r["sinks"] for r in data.get("sham", [])}
    rows = []
    for cond, caps in data.items():
        if cond == "sham": continue
        for r in caps:
            b = base.get((r["prompt_id"], r["seed"]))
            if b is None: continue
            for layer, sinks in r["sinks"].items():
                same = (sinks == b[layer]).float().mean()
                rows.append(dict(condition=cond, layer=layer, prompt_id=r["prompt_id"],
                                 seed=r["seed"], sink_unchanged=float(same),
                                 text_frac=r["text_frac"][layer]))
    df = pd.DataFrame(rows)
    if df.empty:
        print("run sham + at least one intervention first"); return df
    piv = df.pivot_table(index="condition", columns="layer", values="sink_unchanged")
    print(f"E2 [{cfg.model_key}] fraction of heads with UNCHANGED top-1 sink vs sham (final step):")
    print(piv.round(3).to_string())
    print("  seed_clamp HIGH + seed_rotate LOW  ->  direction causes sink selection (replicates FLUX-schnell)")
    df.to_csv(out / "causal_summary.csv", index=False)
    ax = piv.T.plot(kind="bar", figsize=(9, 4))
    ax.set_ylabel("sink unchanged vs sham"); ax.set_ylim(0, 1.05)
    plt.title(f"{cfg.model_key}: magnitude vs direction ablation")
    plt.tight_layout(); plt.savefig(out / "fig_causal.png", dpi=180); plt.show()
    return df




In [ ]:
# [CELL 5] ------------------------------------------------------------
# E3: why does high norm protect direction? Prediction vs measurement
# -------------------------------------------------------------------------
# Theory: block update u rotates a token by  theta ~ |u_perp| / |h|  (small-angle),
# so drift is inversely proportional to magnitude. We record, at each block in the
# register window, |h|, |u| and the realised angle for register vs ordinary tokens,
# then plot predicted vs observed cos(theta).

def run_inertia_check(cfg: RunConfig, adapter, window: Tuple[int, int]) -> None:
    out = cfg.out / "inertia"; out.mkdir(exist_ok=True)
    blocks = [(i, n, b) for i, n, b, k in adapter.iter_blocks() if window[0] <= i <= window[1]]
    state = {"step": -1, "pre": {}, "rows": []}

    def make_pre(idx):
        def pre(m, a, k):
            x = k.get("hidden_states", a[0] if a else None)
            state["pre"][idx] = x[0].detach().float()
            return None
        return pre

    def make_post(idx):
        def post(m, a, k, o):
            h0 = state["pre"].pop(idx, None)
            if h0 is None: return
            h1 = block_post_image(o)[0].detach().float()
            u = h1 - h0
            n0 = h0.norm(dim=-1); n_u = u.norm(dim=-1)
            cos_obs = F.cosine_similarity(h0, h1, dim=-1)
            u_par = (u * h0).sum(-1) / n0.clamp(min=1e-6)
            u_perp = (n_u.pow(2) - u_par.pow(2)).clamp(min=0).sqrt()
            theta_pred = torch.atan2(u_perp, n0 + u_par)
            med = n0.median()
            reg = n0 > cfg.seed_ratio * med
            for grp, mask in [("register", reg), ("ordinary", ~reg)]:
                if mask.sum() == 0: continue
                state["rows"].append(dict(layer=idx, step=state["step"], group=grp,
                                          n=int(mask.sum()),
                                          mean_h=float(n0[mask].mean()),
                                          mean_u=float(n_u[mask].mean()),
                                          cos_pred=float(torch.cos(theta_pred[mask]).mean()),
                                          cos_obs=float(cos_obs[mask].mean())))
        return post

    handles = [adapter.transformer.register_forward_pre_hook(
        lambda m, a, k: state.__setitem__("step", state["step"] + 1), with_kwargs=True)]
    for idx, name, b in blocks:
        handles.append(b.register_forward_pre_hook(make_pre(idx), with_kwargs=True))
        handles.append(b.register_forward_hook(make_post(idx), with_kwargs=True))
    try:
        for pid, prompt in enumerate(cfg.prompts[:2]):
            for seed in cfg.seeds[:2]:
                fp = out / f"inertia_p{pid}_s{seed}.csv"
                if fp.exists(): continue
                state["step"], state["rows"] = -1, []
                adapter.run(prompt, seed)
                pd.DataFrame(state["rows"]).to_csv(fp, index=False)
                print("saved", fp.name)
    finally:
        for h in handles: h.remove()
        torch.cuda.empty_cache()


def analyze_inertia(cfg: RunConfig) -> None:
    out = cfg.out / "inertia"
    df = pd.concat([pd.read_csv(f) for f in sorted(out.glob("inertia_*.csv"))], ignore_index=True)
    g = df.groupby(["group"]).agg(mean_h=("mean_h", "mean"), mean_u=("mean_u", "mean"),
                                  cos_pred=("cos_pred", "mean"), cos_obs=("cos_obs", "mean"))
    g["u_over_h"] = g["mean_u"] / g["mean_h"]
    print(f"E3 [{cfg.model_key}] inertia check (register window):")
    print(g.round(4).to_string())
    print("""  Reading: if mean_u is SIMILAR for both groups but mean_h is ~10x larger for
  registers, then u/h (and hence drift) is ~10x smaller for registers, and
  cos_obs ~ cos_pred confirms the small-angle model: MAGNITUDE = INERTIA, derived
  and measured, closing Q4.""")
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(df["cos_pred"], df["cos_obs"], c=(df["group"] == "register").map({True: "tab:red", False: "tab:blue"}),
               s=10, alpha=0.5)
    lo = min(df["cos_pred"].min(), df["cos_obs"].min())
    ax.plot([lo, 1], [lo, 1], "k--", lw=1)
    ax.set_xlabel("predicted cos(drift) from |u|,|h|"); ax.set_ylabel("observed cos(drift)")
    ax.set_title(f"{cfg.model_key}: geometric drift model vs measurement\n(red=register, blue=ordinary)")
    plt.tight_layout(); plt.savefig(out / "fig_inertia.png", dpi=180); plt.show()


In [ ]:
# [CELL 6] ------------------------------------------------------------
# E4: why is the register key most query-aligned? Weight-level answer
# -------------------------------------------------------------------------
# QGrab keeps its query store SEPARATE from the
# LightSinkProcessor sink dict (the parent was overwriting the tensor with its
# own {"incoming": ...} dict, causing 'dict' object has no attribute 'norm').

@torch.no_grad()
def analytic_qk(cfg: RunConfig, adapter, register_channel: int,
                layers: Optional[List[str]] = None, n_null: int = 64) -> pd.DataFrame:
    out = cfg.out / "qk"; out.mkdir(exist_ok=True)
    if layers is None:
        layers = (["single_0", "single_4", "single_8"] if adapter.KIND == "flux"
                  else [n for _, n, _, _ in adapter.iter_blocks()][15:18])
    name2block = {n: b for _, n, b, _ in adapter.iter_blocks()}

    class QGrab(LightSinkProcessor):
        """Capture per-head mean image-query direction (post q-norm, pre-RoPE)."""
        def __init__(self, qstore, tag, image_slice_fn):
            super().__init__({}, tag, image_slice_fn)      # parent gets a throwaway sink
            self.qstore = qstore
        def __call__(self, attn, hidden_states, encoder_hidden_states=None,
                     attention_mask=None, image_rotary_emb=None, **kw):
            q = attn.to_q(hidden_states)
            B, S, _ = q.shape; H = attn.heads; Dh = q.shape[-1] // H
            qh = q.view(B, S, H, Dh)
            if getattr(attn, "norm_q", None) is not None:
                qh = attn.norm_q(qh)
            i0, i1 = self.image_slice_fn(S)
            self.qstore[self.tag] = qh[0, i0:i1].mean(0).float().cpu()      # [H, Dh]
            return super().__call__(attn, hidden_states, encoder_hidden_states,
                                    attention_mask, image_rotary_emb, **kw)

    qstore, originals = {}, {}
    for n in layers:
        b = name2block[n]
        attn = b.attn if adapter.KIND == "flux" else b.attn1
        originals[n] = attn.processor
        attn.set_processor(QGrab(qstore, n, adapter.image_slice))
    adapter.run(cfg.prompts[0], cfg.seeds[0])
    for n in layers:
        b = name2block[n]
        (b.attn if adapter.KIND == "flux" else b.attn1).set_processor(originals[n])

    rows = []
    for n in layers:
        if n not in qstore:
            print(f"  (no query capture for {n}, skipping)"); continue
        b = name2block[n]
        attn = b.attn if adapter.KIND == "flux" else b.attn1
        C = attn.to_k.in_features; H = attn.heads; Dh = attn.to_k.out_features // H

        def key_of(channel_vec):
            v = channel_vec.to(attn.to_k.weight.dtype).to(attn.to_k.weight.device)
            v = (v - v.mean()) / v.std(unbiased=False).clamp(min=1e-6)   # LayerNorm core
            k = attn.to_k(v.unsqueeze(0)).view(1, H, Dh)
            if getattr(attn, "norm_k", None) is not None:
                k = attn.norm_k(k)
            return F.normalize(k[0].float().cpu(), dim=-1)               # [H, Dh]

        qm = F.normalize(qstore[n], dim=-1)                              # [H, Dh]
        e = torch.zeros(C); e[register_channel] = 1.0
        cos_reg = (key_of(e) * qm).sum(-1)                               # [H]
        null = []
        g = det_rng("qknull", n)
        for c in torch.randperm(C, generator=g)[:n_null].tolist():
            en = torch.zeros(C); en[c] = 1.0
            null.append((key_of(en) * qm).sum(-1))
        null = torch.stack(null)                                         # [n_null, H]
        pct = (null < cos_reg.unsqueeze(0)).float().mean(0)              # percentile per head
        rows.append(dict(layer=n, mean_cos_register=float(cos_reg.mean()),
                         mean_cos_null=float(null.mean()),
                         frac_heads_above_all_null=float((pct == 1.0).float().mean()),
                         mean_percentile=float(pct.mean())))
    df = pd.DataFrame(rows)
    print(f"E4 [{cfg.model_key}] analytic QK: does W_k map channel {register_channel} "
          f"onto the query consensus?")
    print(df.round(3).to_string(index=False))
    print("  mean_percentile ~1.0 -> the preference is IN THE WEIGHTS (prompt-independent), closing Q5.")
    df.to_csv(out / "analytic_qk.csv", index=False)
    return df

In [ ]:
# [CELL 7] ------------------------------------------------------------
# E5: which MLP neurons write the register channel? (birth-block circuit)
# -------------------------------------------------------------------------
# For the birth block's MLP (fc1 -> act -> fc2), rank hidden neurons by
# |fc2.weight[ch, j]|, capture their activations on register vs ordinary tokens,
# then ablate the top-k neurons (zero their fc2 columns) and test whether
# register birth is suppressed at the next layer. A positive result identifies
# the register-writing circuit.

def find_birth_mlp(adapter, birth_layer_idx: int):
    for idx, name, b, kind in adapter.iter_blocks():
        if idx == birth_layer_idx:
            if kind == "dual":
                return name, b.ff.net[0].proj, b.ff.net[2]      # fc1(GELU proj), fc2
            if kind == "basic":
                return name, b.ff.net[0].proj, b.ff.net[2]
            raise ValueError("birth layer should be a dual/basic block")
    raise ValueError("layer not found")


@torch.no_grad()
def register_neurons(cfg: RunConfig, adapter, register_channel: int,
                     birth_layer_idx: int, top_k: int = 32) -> dict:
    out = cfg.out / "neurons"; out.mkdir(exist_ok=True)
    name, fc1, fc2 = find_birth_mlp(adapter, birth_layer_idx)
    w_out = fc2.weight[register_channel].detach().float().cpu()          # [hidden_mlp]
    order = torch.argsort(w_out.abs(), descending=True)
    top = order[:top_k]
    share = float(w_out[top].pow(2).sum() / w_out.pow(2).sum())
    print(f"E5 [{cfg.model_key}] birth block {name}: top-{top_k} neurons carry "
          f"{share:.2%} of fc2 row-{register_channel} energy")

    # capture neuron activations on register vs ordinary tokens for one generation
    acts = {"rows": [], "step": -1, "pre": None}
    def pre_block(m, a, k):
        x = k.get("hidden_states", a[0] if a else None)
        acts["pre"] = x[0].detach().float()
        return None
    def fc1_hook(m, inp, outp):
        h0 = acts["pre"]
        if h0 is None: return
        a_ = outp[0].detach().float()                                     # [N, hidden_mlp] (pre-act proj)
        n0 = h0.norm(dim=-1); med = n0.median()
        reg = n0 > cfg.seed_ratio * med
        for grp, mask in [("register", reg), ("ordinary", ~reg)]:
            if mask.sum() == 0: continue
            acts["rows"].append(dict(step=acts["step"], group=grp,
                                     mean_topk_act=float(a_[mask][:, top].abs().mean()),
                                     mean_other_act=float(a_[mask][:, order[top_k:2000]].abs().mean())))
    blocks = {i: b for i, n, b, k in adapter.iter_blocks()}
    h1 = adapter.transformer.register_forward_pre_hook(
        lambda m, a, k: acts.__setitem__("step", acts["step"] + 1), with_kwargs=True)
    h2 = blocks[birth_layer_idx].register_forward_pre_hook(pre_block, with_kwargs=True)
    h3 = fc1.register_forward_hook(fc1_hook)
    adapter.run(cfg.prompts[0], cfg.seeds[0])
    h1.remove(); h2.remove(); h3.remove()
    act_df = pd.DataFrame(acts["rows"])
    print(act_df.groupby("group")[["mean_topk_act", "mean_other_act"]].mean().round(3).to_string())
    print("  register >> ordinary on top-k neurons -> these neurons fire selectively on the winners.")

    # ablation: zero the top-k fc2 columns for channel ch only? No: zero whole columns
    # is too blunt; we zero ONLY their contribution to the register channel row.
    result = {"name": name, "top_neurons": top.tolist(), "energy_share": share}
    saved = fc2.weight.data[register_channel, top].clone()
    fc2.weight.data[register_channel, top] = 0.0
    try:
        # measure birth at boundary: register count entering the next block
        probe = {"n_reg": []}
        def probe_pre(m, a, k):
            x = k.get("hidden_states", a[0] if a else None)
            n = x[0].float().norm(dim=-1)
            probe["n_reg"].append(int((n > cfg.seed_ratio * n.median()).sum()))
            return None
        hb = adapter.boundary_block().register_forward_pre_hook(probe_pre, with_kwargs=True)
        adapter.run(cfg.prompts[0], cfg.seeds[0])
        hb.remove()
        result["registers_after_neuron_ablation"] = probe["n_reg"]
        print(f"  registers entering boundary AFTER ablating {top_k} neurons (per step): {probe['n_reg']}")
        print("  ~0 everywhere -> these neurons ARE the birth circuit (Q7 closed at neuron level).")
    finally:
        fc2.weight.data[register_channel, top] = saved                    # restore!
    act_df.to_csv(out / "neuron_acts.csv", index=False)
    json.dump({k: v for k, v in result.items() if k != "top_neurons"} |
              {"top_neurons": result["top_neurons"]}, open(out / "neuron_result.json", "w"), indent=2)
    return result




In [ ]:
# [CELL 8] ------------------------------------------------------------
# E6: text tokens (Flux only; PixArt text is cross-attn only)
# -------------------------------------------------------------------------
# (a) Census of the TEXT stream: per-position norms + register-channel value,
#     locating template/padding structure (to compare against 2607.19139).
# (b) Padding-sink masking: zero image->padding attention logits and measure
#     where the mass goes (image register? real prompt tokens?) + image change.

class PaddingMaskProcessor(LightSinkProcessor):
    """Light capture + optional masking of image-query -> padding-key attention."""
    def __init__(self, sink, tag, image_slice_fn, pad_start: int, mask: bool):
        super().__init__(sink, tag, image_slice_fn)
        self.pad_start, self.mask = pad_start, mask

    def __call__(self, attn, hidden_states, encoder_hidden_states=None,
                 attention_mask=None, image_rotary_emb=None, **kw):
        x = hidden_states
        q = attn.to_q(x); k = attn.to_k(x); v = attn.to_v(x)
        B, S, _ = q.shape; H = attn.heads; Dh = q.shape[-1] // H
        q = q.view(B, S, H, Dh); k = k.view(B, S, H, Dh); v = v.view(B, S, H, Dh)
        if getattr(attn, "norm_q", None) is not None: q = attn.norm_q(q)
        if getattr(attn, "norm_k", None) is not None: k = attn.norm_k(k)
        if image_rotary_emb is not None:
            from diffusers.models.embeddings import apply_rotary_emb
            q = apply_rotary_emb(q, image_rotary_emb, sequence_dim=1)
            k = apply_rotary_emb(k, image_rotary_emb, sequence_dim=1)
        qh, kh, vh = (t.permute(0, 2, 1, 3) for t in (q, k, v))
        i0, i1 = self.image_slice_fn(S)
        bias = None
        if self.mask and i0 > 0:
            bias = torch.zeros(1, 1, S, S, device=q.device, dtype=torch.float32)
            bias[:, :, i0:, self.pad_start:i0] = float("-inf")   # image queries -> padding keys
        # light incoming capture with the same bias, chunked
        scale = Dh ** -0.5
        inc = torch.zeros(H, S, device=q.device, dtype=torch.float32); nq = 0
        for s in range(i0, i1, self.chunk):
            e = min(s + self.chunk, i1)
            logits = torch.einsum("bhqd,bhkd->bhqk", qh[:, :, s:e].float(), kh.float()) * scale
            if bias is not None:
                logits = logits + bias[:, :, s:e]
            inc += logits.softmax(-1)[0].sum(1); nq += (e - s)
        inc /= max(nq, 1)
        self.sink[self.tag] = {"incoming": inc[:, i0:].cpu(),
                               "text_incoming": inc[:, :i0].cpu() if i0 > 0 else None}
        out = F.scaled_dot_product_attention(qh, kh, vh,
                                             attn_mask=bias.to(qh.dtype) if bias is not None else None)
        return out.permute(0, 2, 1, 3).reshape(B, S, H * Dh).to(hidden_states.dtype)


def run_text_experiments(cfg: RunConfig, adapter, register_channel: int,
                         layers: Sequence[str] = ("single_0", "single_4", "single_8")) -> None:
    assert adapter.KIND == "flux", "E6 requires text in the self-attn stream (Flux only)"
    out = cfg.out / "text"; out.mkdir(exist_ok=True)
    tok = adapter.pipe.tokenizer_2

    # (a) text-stream census via dual-block encoder states
    rows = []
    def make_dual_post(idx, name):
        def post(m, a, k, o):
            if not (isinstance(o, tuple) and len(o) >= 2): return
            enc = o[0][0].detach().float()                       # [T, C] text stream
            n = enc.norm(dim=-1)
            rows.append(dict(layer=idx, name=name,
                             norms=json.dumps(n[:64].tolist()),
                             x_ch=json.dumps(enc[:64, register_channel].tolist()),
                             argmax_pos=int(n.argmax()), max_over_med=float(n.max() / n.median())))
        return post
    handles = []
    for idx, name, b, kind in adapter.iter_blocks():
        if kind == "dual" and idx in (10, 14, 18):
            handles.append(b.register_forward_hook(make_dual_post(idx, name), with_kwargs=True))
    for pid, prompt in enumerate(cfg.prompts[:3]):
        L = int(tok(prompt, return_tensors="pt", padding="max_length",
                    max_length=adapter.text_length, truncation=True).attention_mask.sum())
        adapter.run(prompt, cfg.seeds[0])
        for r in rows:
            r.setdefault("prompt_id", pid); r.setdefault("prompt_len", L)
    for h in handles: h.remove()
    pd.DataFrame(rows).to_csv(out / "text_census.csv", index=False)
    print("E6a: text census saved (per-position norms + register-channel values; "
          "check whether the max-norm text position is the first padding token, prompt_len).")

    # (b) padding-mask intervention vs baseline
    for mask in (False, True):
        tag = "pad_masked" if mask else "pad_baseline"
        fp = out / f"sinks_{tag}.pt"
        if fp.exists(): continue
        caps = []
        for pid, prompt in enumerate(cfg.prompts[:3]):
            L = int(tok(prompt, return_tensors="pt", padding="max_length",
                        max_length=adapter.text_length, truncation=True).attention_mask.sum())
            sink = {}
            originals = {}
            name2block = {n: b for _, n, b, _ in adapter.iter_blocks()}
            for n in layers:
                originals[n] = name2block[n].attn.processor
                name2block[n].attn.set_processor(
                    PaddingMaskProcessor(sink, n, adapter.image_slice, pad_start=L, mask=mask))
            img = adapter.run(prompt, cfg.seeds[0])
            (out / "images").mkdir(exist_ok=True)
            img.save(out / "images" / f"{tag}_p{pid}.png")
            for n in layers:
                name2block[n].attn.set_processor(originals[n])
            caps.append(dict(prompt_id=pid, prompt_len=L,
                             text_frac={n: float(d["text_incoming"].sum(-1).mean()) for n, d in sink.items()},
                             img_sink_strength={n: float(d["incoming"].max(-1).values.mean())
                                                for n, d in sink.items()}))
        torch.save(caps, fp); print("saved", tag)

    b0 = torch.load(out / "sinks_pad_baseline.pt", weights_only=False)
    b1 = torch.load(out / "sinks_pad_masked.pt", weights_only=False)
    tf0 = np.mean([v for r in b0 for v in r["text_frac"].values()])
    tf1 = np.mean([v for r in b1 for v in r["text_frac"].values()])
    s0 = np.mean([v for r in b0 for v in r["img_sink_strength"].values()])
    s1 = np.mean([v for r in b1 for v in r["img_sink_strength"].values()])
    print(f"E6b: text attention fraction {tf0:.3f} -> {tf1:.3f} under padding mask; "
          f"image sink strength {s0:.4f} -> {s1:.4f}")
    print("  strength UP under mask -> the image register absorbs the displaced mass "
          "(substitution confirmed from the text side).")




In [ ]:
# [CELL 8b] -----------------------------------------------------------
# E6b: is FLUX's padding sink SEMANTIC (Li et al. 2607.19139) or a NO-OP?
# Three conditions, mirroring their causal framework:
#   pad_value_zero : zero the V vectors of padding keys for image queries.
#                    Attention still flows to padding but transmits nothing.
#                    Object changes  -> padding carries content (SEMANTIC).
#                    Object intact   -> padding is a no-op sink (like ch-154).
#   pad_read_mask  : mask PADDING-QUERIES -> IMAGE-KEYS (their register-read test).
#                    If padding acquires semantics by reading the image (their
#                    mechanism), this should degrade identity like their Fig.5b.
#   pad_swap       : replace padding hidden states with another prompt's padding
#                    states at the dual->single boundary (their cross-prompt swap).
# Compare all against sham via image similarity + qualitative grid.

class PaddingSemanticsProcessor(LightSinkProcessor):
    def __init__(self, sink, tag, image_slice_fn, pad_start, mode):
        super().__init__(sink, tag, image_slice_fn)
        self.pad_start, self.mode = pad_start, mode      # mode in {none, value_zero, read_mask}

    def __call__(self, attn, hidden_states, encoder_hidden_states=None,
                 attention_mask=None, image_rotary_emb=None, **kw):
        x = hidden_states
        q = attn.to_q(x); k = attn.to_k(x); v = attn.to_v(x)
        B, S, _ = q.shape; H = attn.heads; Dh = q.shape[-1] // H
        q = q.view(B, S, H, Dh); k = k.view(B, S, H, Dh); v = v.view(B, S, H, Dh)
        if getattr(attn, "norm_q", None) is not None: q = attn.norm_q(q)
        if getattr(attn, "norm_k", None) is not None: k = attn.norm_k(k)
        if image_rotary_emb is not None:
            from diffusers.models.embeddings import apply_rotary_emb
            q = apply_rotary_emb(q, image_rotary_emb, sequence_dim=1)
            k = apply_rotary_emb(k, image_rotary_emb, sequence_dim=1)
        i0, _ = self.image_slice_fn(S)          # text length
        p0 = self.pad_start                     # first padding position
        qh, kh, vh = (t.permute(0, 2, 1, 3) for t in (q, k, v))
        bias = None
        if self.mode == "value_zero" and i0 > 0:
            # zero padding VALUES only for the whole sequence: padding keys keep
            # absorbing attention, but transmit nothing (no-op-ification).
            vh = vh.clone(); vh[:, :, p0:i0, :] = 0.0
        elif self.mode == "read_mask" and i0 > 0:
            # padding QUERIES may not read IMAGE keys (their register-read test)
            bias = torch.zeros(1, 1, S, S, device=q.device, dtype=torch.float32)
            bias[:, :, p0:i0, i0:] = float("-inf")
        out = F.scaled_dot_product_attention(
            qh, kh, vh, attn_mask=bias.to(qh.dtype) if bias is not None else None)
        return out.permute(0, 2, 1, 3).reshape(B, S, H * Dh).to(hidden_states.dtype)


class PadSwapHook:
    """Replace padding-token hidden states in the TEXT stream at each dual block
    input with donor states captured from another prompt's run."""
    def __init__(self, donor_states, pad_start):
        self.donor, self.p0, self.enabled = donor_states, pad_start, True
        self.step = -1
    def __call__(self, module, args, kwargs):
        if not self.enabled: return None
        enc = kwargs.get("encoder_hidden_states", None)
        key = (id(module), self.step)
        if enc is None or key not in self.donor: return None
        enc = enc.clone()
        d = self.donor[key]
        enc[0, self.p0:self.p0 + d.shape[0]] = d.to(enc.dtype).to(enc.device)
        kwargs = dict(kwargs); kwargs["encoder_hidden_states"] = enc
        return (args, kwargs)


def run_padding_semantics(cfg, adapter, layers=("single_0", "single_4", "single_8")):
    assert adapter.KIND == "flux"
    out = cfg.out / "text" / "semantics"; out.mkdir(parents=True, exist_ok=True)
    tok = adapter.pipe.tokenizer_2
    name2block = {n: b for _, n, b, _ in adapter.iter_blocks()}
    for mode in ("none", "value_zero", "read_mask"):
        d = out / "images" / mode; d.mkdir(parents=True, exist_ok=True)
        for pid, prompt in enumerate(cfg.prompts[:3]):
            fp = d / f"p{pid}.png"
            if fp.exists(): continue
            L = int(tok(prompt, return_tensors="pt", padding="max_length",
                        max_length=adapter.text_length, truncation=True).attention_mask.sum())
            sink, orig = {}, {}
            # apply to ALL single blocks (the effect they see is cumulative over depth)
            for _, n, b, kind in adapter.iter_blocks():
                if kind != "single": continue
                orig[n] = b.attn.processor
                b.attn.set_processor(PaddingSemanticsProcessor(
                    sink, n, adapter.image_slice, pad_start=L, mode=mode))
            img = adapter.run(prompt, cfg.seeds[0])
            img.save(fp)
            for n, p in orig.items(): name2block[n].attn.set_processor(p)
            print(f"saved {mode}/p{pid}")
    print("Compare images/none vs images/value_zero (SEMANTIC test) and images/read_mask "
          "(their register-read mechanism). Object identity destroyed under value_zero "
          "-> padding carries semantics; intact -> padding is a no-op sink.")

In [ ]:
# [CELL 9] ------------------------------------------------------------
# Reporting: one row per model for a cross-model summary table
# -------------------------------------------------------------------------
def paper_table() -> pd.DataFrame:
    rows = []
    for key in ["flux-schnell", "flux-dev", "pixart-sigma"]:
        d = DRIVE_ROOT / key
        r = {"model": key}
        cs = d / "census" / "census_summary.pt"
        if cs.exists():
            s = torch.load(cs, weights_only=False)
            r |= {"register_channel": s.get("register_channel"),
                  "top1_energy": round(s.get("top1_energy", float("nan")), 3),
                  "spike_to_floor": round(s.get("spike_to_floor", float("nan")), 1)}
        cc = d / "causal" / "causal_summary.csv"
        if cc.exists():
            c = pd.read_csv(cc)
            for cond in ["seed_clamp", "seed_rotate"]:
                sub = c[c.condition == cond]
                if len(sub):
                    r[f"{cond}_sink_unchanged"] = round(sub["sink_unchanged"].mean(), 3)
        rows.append(r)
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    df.to_csv(DRIVE_ROOT / "cross_model_table.csv", index=False)
    return df


## Run driver
Recommended order per model (each section is resume-safe):
1. `E1` census → `analyze_census` (gives `register_channel` for everything downstream)
2. `E2` causal core → `analyze_causal` (the headline replication)
3. `E3` inertia, `E4` analytic QK, `E5` register neurons
4. `E6` text experiments (Flux models only)
5. `paper_table()` after any subset, aggregates whatever exists

Costs at 512²: E1 ≈ 15 gens · E2 ≈ 45 gens · E3 ≈ 4 gens · E4/E5 ≈ 1 to 2 gens · E6 ≈ 9 gens.
For flux-dev, consider `capture_steps=[0, 13, 27]` (default) and the same prompts/seeds as schnell so results are directly comparable.


In [ ]:
# [CELL 10] -----------------------------------------------------------
# Driver: set MODEL_KEY and run sections top to bottom
# -------------------------------------------------------------------------
MODEL_KEY = "flux-schnell"          # "flux-schnell" | "flux-dev" | "pixart-sigma"

cfg = RunConfig(model_key=MODEL_KEY)
cfg.stamp()
adapter = make_adapter(cfg)
print(f"{MODEL_KEY}: {adapter.default_steps} steps, guidance {adapter.guidance}, "
      f"{cfg.height}x{cfg.width} -> {adapter.n_image_tokens()} image tokens")

# --- E1: census + register-channel identity ---
run_census(cfg, adapter)
census = analyze_census(cfg)
REGISTER_CHANNEL = census["register_channel"]

# --- E2: causal core (sham / seed_clamp / seed_rotate) ---
run_causal(cfg, adapter)
analyze_causal(cfg)

# --- E3: inertia. Window = the register zone found by E1 (Flux: 19-35) ---
run_inertia_check(cfg, adapter, window=(19, 35) if adapter.KIND == "flux" else (14, 20))
analyze_inertia(cfg)

# --- E4: analytic QK ---
analytic_qk(cfg, adapter, REGISTER_CHANNEL)

# --- E5: register neurons. Birth layer: Flux 18; PixArt: first census spike ---
register_neurons(cfg, adapter, REGISTER_CHANNEL, birth_layer_idx=18 if adapter.KIND == "flux" else 13)

# --- E6: text experiments (Flux only) ---
if adapter.KIND == "flux":
     run_text_experiments(cfg, adapter, REGISTER_CHANNEL)
     # E6b: padding semantics
     run_padding_semantics(cfg, adapter)

# --- Cross-model paper table ---
paper_table()


In [ ]:
# --- E5: register neurons. Birth layer: Flux 18; PixArt: first census spike ---
register_neurons(cfg, adapter, REGISTER_CHANNEL, birth_layer_idx=18 if adapter.KIND == "flux" else 13)

# --- E6: text experiments (Flux only) ---
if adapter.KIND == "flux":
     run_text_experiments(cfg, adapter, REGISTER_CHANNEL)
     # E6b: padding semantics
     run_padding_semantics(cfg, adapter)

# --- Cross-model paper table ---
paper_table()

In [ ]:
# --- E4: analytic QK ---
analytic_qk(cfg, adapter, REGISTER_CHANNEL)

In [ ]:
# ================================================================
# MOUNT REAL GOOGLE DRIVE + SAVE ALL EXISTING RESULTS TO IT
# ================================================================

from google.colab import drive
from pathlib import Path
import shutil
import os

# ------------------------------------------------
# 1. Your CURRENT results in Colab
#    DO NOT delete this folder.
# ------------------------------------------------

LOCAL_RESULTS = Path("/content/drive/MyDrive/sparse_structures_repro")

print("=" * 70)
print("1. CHECKING EXISTING RESULTS")
print("=" * 70)

if not LOCAL_RESULTS.exists():
    raise RuntimeError(
        f"Could not find your existing results at:\n{LOCAL_RESULTS}"
    )

local_files = [p for p in LOCAL_RESULTS.rglob("*") if p.is_file()]

print(f"Found {len(local_files)} existing result files.")
print(f"Current results location:\n{LOCAL_RESULTS}")


# ------------------------------------------------
# 2. Mount the REAL Google Drive somewhere new.
#    We intentionally DO NOT mount at /content/drive
#    because that currently contains your local results.
# ------------------------------------------------

MOUNT_POINT = Path("/content/real_google_drive")

# If this name somehow already contains ordinary files,
# choose another fresh mount point rather than deleting anything.
counter = 2
while (
    MOUNT_POINT.exists()
    and any(MOUNT_POINT.iterdir())
    and not (MOUNT_POINT / "MyDrive").exists()
):
    MOUNT_POINT = Path(f"/content/real_google_drive_{counter}")
    counter += 1

print("\n" + "=" * 70)
print("2. MOUNTING REAL GOOGLE DRIVE")
print("=" * 70)
print(f"Mount point: {MOUNT_POINT}")
print("Authorize the Google account whose Drive you want to use.\n")

# Mount only if it isn't already mounted there
if not (MOUNT_POINT / "MyDrive").exists():
    drive.mount(str(MOUNT_POINT))
else:
    print("Google Drive already appears to be mounted here.")


# ------------------------------------------------
# 3. Verify this really exposes MyDrive
# ------------------------------------------------

REAL_MYDRIVE = MOUNT_POINT / "MyDrive"

if not REAL_MYDRIVE.exists():
    raise RuntimeError(
        "Google Drive did not mount correctly: MyDrive was not found."
    )

print("\nGoogle Drive mounted successfully:")
print(REAL_MYDRIVE)


# ------------------------------------------------
# 4. Copy ALL current results into real Google Drive
# ------------------------------------------------

REAL_RESULTS = REAL_MYDRIVE / "sparse_structures_repro"

print("\n" + "=" * 70)
print("3. COPYING RESULTS TO GOOGLE DRIVE")
print("=" * 70)

print("FROM:")
print(LOCAL_RESULTS)

print("\nTO:")
print(REAL_RESULTS)

# dirs_exist_ok=True means it is safe to run again:
# existing files/folders will be merged rather than deleted.
shutil.copytree(
    LOCAL_RESULTS,
    REAL_RESULTS,
    dirs_exist_ok=True
)

# Ask OS to flush pending writes
os.sync()


# ------------------------------------------------
# 5. Verify all files were copied
# ------------------------------------------------

drive_files = [p for p in REAL_RESULTS.rglob("*") if p.is_file()]

print("\n" + "=" * 70)
print("4. VERIFYING COPY")
print("=" * 70)

print(f"Files originally in Colab : {len(local_files)}")
print(f"Files now in Google Drive : {len(drive_files)}")

if len(drive_files) < len(local_files):
    print("\nWARNING: Google Drive contains fewer files than the local folder.")
    print("Do NOT restart Colab yet.")
else:
    print("\n✓ All existing result files appear to have been copied.")


# ------------------------------------------------
# 6. Create an obvious verification file
# ------------------------------------------------

TEST_FILE = REAL_RESULTS / "GOOGLE_DRIVE_SAVE_CONFIRMED.txt"

TEST_FILE.write_text(
    "Sparse Structures experiment results were successfully copied "
    "from Colab to Google Drive.\n"
)

os.sync()

print(f"\nVerification file created:\n{TEST_FILE}")


# ------------------------------------------------
# 7. IMPORTANT:
#    Redirect ALL FUTURE notebook output to real Drive
# ------------------------------------------------

DRIVE_ROOT = REAL_RESULTS

print("\n" + "=" * 70)
print("5. FUTURE SAVING LOCATION UPDATED")
print("=" * 70)

print("DRIVE_ROOT =", DRIVE_ROOT)

print(
    "\nFrom now on, anything using cfg.out will save under the "
    "REAL Google Drive."
)


# ------------------------------------------------
# 8. Show the important files that made it across
# ------------------------------------------------

important = [
    "cross_model_table.csv",
    "flux-schnell/census/fig_census.png",
    "flux-schnell/causal/fig_causal.png",
    "flux-schnell/inertia/fig_inertia.png",
    "flux-schnell/neurons/neuron_result.json",
    "flux-schnell/qk/analytic_qk.csv",
    "flux-schnell/text/text_census.csv",
    "flux-schnell/text/semantics/images/none/p0.png",
    "flux-schnell/text/semantics/images/value_zero/p0.png",
    "flux-schnell/text/semantics/images/read_mask/p0.png",
]

print("\n" + "=" * 70)
print("6. IMPORTANT RESULT CHECK")
print("=" * 70)

for rel in important:
    p = REAL_RESULTS / rel
    print(("✓" if p.exists() else "✗"), rel)


print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print(
    "\nNow open Google Drive in your browser and look for:\n\n"
    "My Drive → sparse_structures_repro\n\n"
    "You should also be able to search for:\n"
    "GOOGLE_DRIVE_SAVE_CONFIRMED.txt\n"
)

In [ ]:
# [CELL 11] -----------------------------------------------------------
# E7: channel ablation, subtraction vs causal, restoration, and 154 vs 1446
# -------------------------------------------------------------------------
# Three questions, one capture pass + one small intervention sweep:
#
#   T1  SUBTRACTION vs CAUSAL at the same layer.
#       subtraction = recompute the norm without channel c (bookkeeping only,
#                     model untouched);  causal = zero channel c in the residual
#                     stream and let the model keep running.
#       Prediction: subtraction removes the spikes at every layer (a register is
#       almost pure ch-154), while causal ablation only removes them outside the
#       maintenance window.
#
#   T2  DEPTH PROFILE of causal ablation: at which layer L does zeroing ch-154
#       still suppress registers downstream?  Expected: works before birth (<=17),
#       fails inside the maintained zone (18-30), works again as maintenance
#       fades (>30).  This profile IS a readout of the controller's strength.
#
#   T3  RESTORATION: after zeroing ch-154 at layer L, track x154 on those exact
#       tokens at L+1, L+2, L+4, L+8.  Regrowth = the homeostat repairing the
#       channel, caught in the act.
#
#   Plus: 154 vs 1446 contrast (spike channel vs floor channel) under both modes.
#
# Requires cells 1-2 (RunConfig, adapters, block_post_image) from the repro
# notebook.  Model-agnostic: pass whichever channels E1 reported.
#
# Cost: T1/T3 ~ 1 generation per (layer, channel, mode); T2 sweeps layers.
# At 512^2 / 4 steps this is a few minutes per condition on an A100.

from dataclasses import dataclass

@dataclass
class ChannelAblationConfig:
    channels: Tuple[int, ...] = (154, 1446)
    ablate_layers: Tuple[int, ...] = (15, 17, 18, 20, 24, 28, 30, 34, 38)
    probe_offsets: Tuple[int, ...] = (1, 2, 4, 8)     # for restoration tracking
    observe_layers: Tuple[int, ...] = (19, 21, 24, 28, 32, 36, 40)
    spike_ratio: float = 3.0                          # register = norm > ratio x median


# ---------------------------------------------------------------- T1 ----
@torch.no_grad()
def run_subtraction_scan(cfg: RunConfig, adapter, ab: ChannelAblationConfig,
                         prompt_id: int = 0, seed: Optional[int] = None) -> pd.DataFrame:
    """No intervention. At every block, compute the outlier ratio three ways:
       full norm, norm without channel c, and norm without the top-3 register
       channels.  This is the BOOKKEEPING control for the causal sweep."""
    seed = cfg.seeds[0] if seed is None else seed
    out = cfg.out / "chan_ablation"; out.mkdir(parents=True, exist_ok=True)
    rows, state = [], {"step": -1}

    def make_post(idx, name):
        def post(m, a, k, o):
            h = block_post_image(o)[0].float()                 # [N, C]
            n_full = h.norm(dim=-1)
            med_full = n_full.median()
            rec = dict(layer=idx, name=name, step=state["step"],
                       ratio_full=float(n_full.max() / med_full),
                       n_reg_full=int((n_full > ab.spike_ratio * med_full).sum()))
            # ABSOLUTE per-channel metrics: immune to the floor confound that
            # makes max/median rise when a floor channel is removed.
            for c in ab.channels:
                xc = h[:, c].abs()
                rec[f"x{c}_max"] = float(xc.max())
                rec[f"x{c}_median"] = float(xc.median())
                rec[f"x{c}_spike_to_floor"] = float(xc.max() / xc.median().clamp(min=1e-6))
            # masked (exact) removal, avoids catastrophic cancellation on ~pure tokens
            def _minus(idx):
                keep = torch.ones(h.shape[-1], dtype=torch.bool, device=h.device)
                keep[idx.to(h.device)] = False
                return h[:, keep].norm(dim=-1)
            for c in ab.channels:
                n_c = _minus(torch.tensor([c]))
                med_c = n_c.median()
                rec[f"ratio_minus_{c}"] = float(n_c.max() / med_c)
                rec[f"n_reg_minus_{c}"] = int((n_c > ab.spike_ratio * med_c).sum())
            # BOTH top-k selection rules, because they disagree before layer 18
            for rule, score in (("global", h.abs().mean(0)),
                                ("loudest", h[n_full.argmax()].abs())):
                top3 = torch.topk(score, 3).indices
                n_t3 = _minus(top3)
                rec[f"ratio_minus_top3_{rule}"] = float(n_t3.max() / n_t3.median())
                rec[f"top3_{rule}"] = json.dumps(top3.tolist())
            rec["ratio_minus_top3"] = rec["ratio_minus_top3_loudest"]   # back-compat
            rows.append(rec)
        return post

    handles = [adapter.transformer.register_forward_pre_hook(
        lambda m, a, k: state.__setitem__("step", state["step"] + 1), with_kwargs=True)]
    for idx, name, b, kind in adapter.iter_blocks():
        handles.append(b.register_forward_hook(make_post(idx, name), with_kwargs=True))
    try:
        adapter.run(cfg.prompts[prompt_id], seed)
    finally:
        for h in handles: h.remove()
        torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv(out / f"subtraction_scan_p{prompt_id}_s{seed}.csv", index=False)
    last = df[df["step"] == df["step"].max()]
    print("T1 SUBTRACTION SCAN (bookkeeping only, model untouched) — final step")
    cols = ["layer", "ratio_full"] + [f"ratio_minus_{c}" for c in ab.channels] + ["ratio_minus_top3"]
    print(last[cols].round(2).to_string(index=False))
    print("""  Reading:
    ratio_minus_154 collapses to ~1 inside the register zone -> a register is
      almost pure channel 154; removing it (on paper) leaves it below average.
    ratio_minus_1446 ~ ratio_full (or slightly HIGHER) -> 1446 is a FLOOR channel:
      it shrinks every token proportionally, so the outlier ratio is preserved.""")
    return df


# ---------------------------------------------------------------- T2/T3 ----
class ChannelZeroHook:
    """Zero given channels of the image stream at one block's input, every step.
    Records the affected tokens so restoration can be tracked downstream."""

    def __init__(self, channels: Sequence[int], spike_ratio: float = 3.0):
        self.channels = list(channels)
        self.spike_ratio = spike_ratio
        self.step = -1
        self.targets: Dict[int, torch.Tensor] = {}     # step -> token ids that were registers
        self.enabled = True

    def __call__(self, module, args, kwargs):
        if not self.enabled:
            return None
        in_kw = isinstance(kwargs, dict) and "hidden_states" in kwargs
        x = kwargs["hidden_states"] if in_kw else (args[0] if args else None)
        if x is None:
            return None
        n = x[0].float().norm(dim=-1)
        self.targets[self.step] = torch.nonzero(n > self.spike_ratio * n.median()).flatten().cpu()
        x = x.clone()
        for c in self.channels:
            x[0, :, c] = 0.0
        if in_kw:
            kwargs = dict(kwargs); kwargs["hidden_states"] = x
            return (args, kwargs)
        return ((x,) + tuple(args[1:]), kwargs)


@torch.no_grad()
def run_causal_ablation_sweep(cfg: RunConfig, adapter, ab: ChannelAblationConfig,
                              prompt_id: int = 0, seed: Optional[int] = None,
                              top3_channels: Optional[Sequence[int]] = None) -> pd.DataFrame:
    """T2 + T3. For each (ablation layer, channel set): zero the channel(s) at that
    layer every step, then measure downstream (a) register counts at observation
    layers, (b) x_c on the originally-targeted tokens (restoration)."""
    seed = cfg.seeds[0] if seed is None else seed
    out = cfg.out / "chan_ablation"; out.mkdir(parents=True, exist_ok=True)
    blocks = {idx: b for idx, n, b, k in adapter.iter_blocks()}
    names = {idx: n for idx, n, b, k in adapter.iter_blocks()}
    max_layer = max(blocks)

    channel_sets = [("none", ()),
                    *[(f"ch{c}", (c,)) for c in ab.channels],
                    ("top3", None)]                   # top3 resolved at runtime per capture
    rows = []

    for ab_layer in (-1,) + tuple(ab.ablate_layers):   # -1 = clean baseline
        for set_name, chans in channel_sets:
            if ab_layer == -1 and set_name != "none":
                continue
            if ab_layer != -1 and set_name == "none":
                continue
            if set_name == "top3":
                if top3_channels is None:
                    print("  (skipping top3: pass top3_channels= from the subtraction scan)")
                    continue
                chans = tuple(top3_channels)
            tag = f"L{ab_layer}_{set_name}"
            fp = out / f"causal_{tag}_p{prompt_id}_s{seed}.csv"
            if fp.exists():
                rows.extend(pd.read_csv(fp).to_dict("records")); print("skip", tag); continue

            hook = ChannelZeroHook(chans, ab.spike_ratio) if ab_layer >= 0 else None
            state = {"step": -1, "rec": []}

            def make_obs(idx, name):
                def post(m, a, k, o):
                    h = block_post_image(o)[0].float()
                    n = h.norm(dim=-1); med = n.median()
                    r = dict(ablate_layer=ab_layer, channels=set_name, obs_layer=idx,
                             step=state["step"], ratio=float(n.max() / med),
                             n_registers=int((n > ab.spike_ratio * med).sum()),
                             median_norm=float(med), prompt_id=prompt_id, seed=seed)
                    # PRIMARY metric: did a spike on the register channel survive?
                    # Absolute, so removing a floor channel cannot inflate it.
                    for c in ab.channels:
                        xc = h[:, c].abs()
                        r[f"x{c}_max"] = float(xc.max())
                        r[f"x{c}_median"] = float(xc.median())
                        r[f"x{c}_spike_to_floor"] = float(xc.max() / xc.median().clamp(min=1e-6))
                    # restoration: x_c on the tokens that were registers at the ablation site
                    if hook is not None and state["step"] in hook.targets:
                        tg = hook.targets[state["step"]]
                        if tg.numel():
                            for c in (chans or ()):
                                r[f"x{c}_on_targets"] = float(h[tg, c].abs().mean())
                            r["norm_on_targets"] = float(n[tg].mean())
                            r["norm_ratio_on_targets"] = float(n[tg].mean() / med)
                    state["rec"].append(r)
                return post

            handles = [adapter.transformer.register_forward_pre_hook(
                lambda m, a, k: (state.__setitem__("step", state["step"] + 1),
                                 hook and setattr(hook, "step", state["step"]), None)[-1],
                with_kwargs=True)]
            if hook is not None and ab_layer in blocks:
                handles.append(blocks[ab_layer].register_forward_pre_hook(hook, with_kwargs=True))
            obs = [l for l in ab.observe_layers if l <= max_layer] + \
                  [min(ab_layer + o, max_layer) for o in ab.probe_offsets if ab_layer >= 0]
            for l in sorted(set(obs)):
                if l in blocks:
                    handles.append(blocks[l].register_forward_hook(make_obs(l, names[l]), with_kwargs=True))
            try:
                adapter.run(cfg.prompts[prompt_id], seed)
            finally:
                for h in handles: h.remove()
                torch.cuda.empty_cache()
            pd.DataFrame(state["rec"]).to_csv(fp, index=False)
            rows.extend(state["rec"])
            print("saved", tag)

    df = pd.DataFrame(rows)
    df.to_csv(out / f"causal_sweep_p{prompt_id}_s{seed}.csv", index=False)
    return df


def analyze_channel_ablation(cfg: RunConfig, ab: ChannelAblationConfig,
                             prompt_id: int = 0, seed: Optional[int] = None) -> None:
    seed = cfg.seeds[0] if seed is None else seed
    out = cfg.out / "chan_ablation"
    sub = pd.read_csv(out / f"subtraction_scan_p{prompt_id}_s{seed}.csv")
    cau = pd.read_csv(out / f"causal_sweep_p{prompt_id}_s{seed}.csv")
    last_step = cau["step"].max()
    cau = cau[cau["step"] == last_step]

    # ---- T2: does causal ablation stick?  measured just downstream of the site ----
    base = cau[cau["ablate_layer"] == -1].set_index("obs_layer")["n_registers"]
    rows = []
    for (L, cset), g in cau[cau["ablate_layer"] >= 0].groupby(["ablate_layer", "channels"]):
        dn = g[g["obs_layer"] > L]
        if dn.empty:
            continue
        near = dn.sort_values("obs_layer").iloc[0]
        rows.append(dict(ablate_layer=L, channels=cset, obs_layer=int(near["obs_layer"]),
                         n_reg_after=int(near["n_registers"]),
                         n_reg_baseline=int(base.get(near["obs_layer"], np.nan)),
                         ratio_after=round(float(near["ratio"]), 2)))
    prof = pd.DataFrame(rows).sort_values(["channels", "ablate_layer"])
    print("T2 CAUSAL ABLATION DEPTH PROFILE (registers surviving just downstream):")
    print(prof.to_string(index=False))
    print("""  Reading: suppression works where the MAINTENANCE LOOP is weak or absent.
    works before birth (<=17)      -> nothing to catapult
    fails inside 18-30             -> the homeostat rebuilds the channel
    works again after ~30          -> maintenance has faded, register already decaying
  The profile is therefore a direct readout of the controller's operating range.""")

    # ---- T3: restoration curves ----
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for c in ab.channels:
        col = f"x{c}_on_targets"
        if col not in cau.columns:
            continue
        for L in sorted(cau[cau["channels"] == f"ch{c}"]["ablate_layer"].unique()):
            if L < 0: continue
            g = cau[(cau["ablate_layer"] == L) & (cau["channels"] == f"ch{c}")]
            g = g[g["obs_layer"] > L].sort_values("obs_layer")
            if len(g) > 1:
                axes[0].plot(g["obs_layer"] - L, g[col], marker="o", label=f"ch{c} @ L{L}")
    axes[0].set_xlabel("blocks after the ablation site"); axes[0].set_ylabel("|x_c| on targeted tokens")
    axes[0].set_title("T3: does the model rebuild the zeroed channel?")
    axes[0].legend(fontsize=7)

    s = sub[sub["step"] == sub["step"].max()]
    axes[1].semilogy(s["layer"], s["ratio_full"], label="full norm", marker=".")
    for c in ab.channels:
        axes[1].semilogy(s["layer"], s[f"ratio_minus_{c}"], label=f"minus ch{c}", marker=".")
    axes[1].semilogy(s["layer"], s["ratio_minus_top3"], label="minus top-3", marker=".", ls="--")
    axes[1].axhline(ab.spike_ratio, color="gray", ls=":")
    axes[1].set_xlabel("layer"); axes[1].set_ylabel("max/median norm")
    axes[1].set_title("T1: subtraction (bookkeeping) — which channel carries the spike?")
    axes[1].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(out / "fig_channel_ablation.png", dpi=180); plt.show()

    # ---- headline contrast ----
    print("\n154 vs 1446 SUMMARY (final step, register zone means):")
    zone = s[(s["layer"] >= 19) & (s["layer"] <= 30)]
    for c in ab.channels:
        print(f"  subtraction: ratio_full {zone['ratio_full'].mean():.1f} -> "
              f"minus ch{c} {zone[f'ratio_minus_{c}'].mean():.1f}")
    for c in ab.channels:
        p = prof[prof["channels"] == f"ch{c}"]
        inside = p[(p["ablate_layer"] >= 18) & (p["ablate_layer"] <= 30)]["n_reg_after"].mean()
        outside = p[(p["ablate_layer"] < 18) | (p["ablate_layer"] > 30)]["n_reg_after"].mean()
        print(f"  causal ch{c}: registers after ablation — inside zone {inside:.1f}, outside {outside:.1f}")


# ---- run ----
ab = ChannelAblationConfig(channels=(REGISTER_CHANNEL, 1446))
run_subtraction_scan(cfg, adapter, ab)
run_causal_ablation_sweep(cfg, adapter, ab)
analyze_channel_ablation(cfg, ab)

In [ ]:
# [CELL 12] -----------------------------------------------------------
# E8: spatial maps, reproduce the "top-k channel removal" figure under
#      explicit, controlled definitions, so subtraction and causal ablation
#      can be compared side by side in the same visual language.
# -------------------------------------------------------------------------
# Figure format (matches the earlier figure layout):
#   [generated image | isolated top-1 channel | full norm | minus top-1
#    | minus top-5 | minus top-10 | minus top-20]
#
# Four knobs are made EXPLICIT because each can flip the conclusion:
#   1. mode        : "subtract" (bookkeeping, model untouched)
#                    "causal"   (zero the channels in the residual stream, keep running)
#   2. select      : how the top-k channels are chosen
#                    "global"        rank by token-mean |x| at this layer  (landscape metric)
#                    "loudest_token" rank by |x| of the max-norm token
#                    "fixed"         a supplied list, e.g. [154, 1446, ...]
#   3. layer index : global depth index (dual 0-18, single 19-56 for FLUX).
#                    An earlier notebook's "layer 17" may be single_17 = global 36.
#   4. colour      : shared_scale=True uses ONE colour scale across the row so a
#                    change in magnitude cannot masquerade as a change in pattern.
#
# Requires cells 1-2 of the repro notebook (RunConfig, adapters, block_post_image).

from PIL import Image as PILImage

SELECT_MODES = ("global", "loudest_token", "fixed")


def _topk_channels(h: torch.Tensor, select: str, k: int,
                   fixed: Optional[Sequence[int]] = None) -> torch.Tensor:
    """h: [N, C] float. Returns indices of the k channels to remove."""
    if select == "fixed":
        assert fixed is not None, "select='fixed' needs a channel list"
        return torch.tensor(list(fixed)[:k], dtype=torch.long)
    if select == "global":
        score = h.abs().mean(0)                      # token-mean |x| per channel
    elif select == "loudest_token":
        score = h[h.norm(dim=-1).argmax()].abs()     # profile of the loudest token
    else:
        raise ValueError(select)
    return torch.topk(score, k).indices


def _norm_minus(h: torch.Tensor, ch: torch.Tensor) -> torch.Tensor:
    """Norm with the listed channels removed, computed by MASKING rather than by
    subtracting squares. The subtract-then-sqrt form suffers catastrophic
    cancellation exactly where it matters: when one channel carries ~99% of a
    token's energy (a register), sum(x^2) - x_c^2 subtracts two nearly-equal
    large numbers and loses precision. Masking is exact."""
    keep = torch.ones(h.shape[-1], dtype=torch.bool, device=h.device)
    keep[ch.to(h.device)] = False
    return h[:, keep].norm(dim=-1)


def _grid(v: torch.Tensor, cfg: RunConfig) -> np.ndarray:
    g = cfg.height // 16
    return v[: g * g].reshape(g, g).cpu().numpy()


# ------------------------------------------------------------------ capture
@torch.no_grad()
def capture_spatial_maps(cfg: RunConfig, adapter, layers: Sequence[int],
                         prompt_ids: Sequence[int] = (0, 1, 2),
                         seed: Optional[int] = None,
                         select: str = "global",
                         ks: Sequence[int] = (1, 5, 10, 20),
                         fixed_channels: Optional[Sequence[int]] = None,
                         step: Optional[int] = None) -> Path:
    """SUBTRACTION mode. One clean generation per prompt; computes every map inside
    the hook so only small arrays are stored."""
    seed = cfg.seeds[0] if seed is None else seed
    out = cfg.out / "spatial" / f"subtract_{select}"; out.mkdir(parents=True, exist_ok=True)
    target_step = (adapter.default_steps - 1) if step is None else step
    store, state = {}, {"step": -1}

    def make_post(idx):
        def post(m, a, k, o):
            if state["step"] != target_step or idx not in layers:
                return
            h = block_post_image(o)[0].float()
            n_full = h.norm(dim=-1)
            rec = {"full": _grid(n_full, cfg), "channels": {}}
            for k_ in ks:
                ch = _topk_channels(h, select, k_, fixed_channels)
                rec[f"minus{k_}"] = _grid(_norm_minus(h, ch), cfg)
                rec["channels"][k_] = ch.tolist()
            top1 = _topk_channels(h, select, 1, fixed_channels)[0]
            rec["isolated"] = _grid(h[:, top1].abs(), cfg)
            rec["top1_channel"] = int(top1)
            rec["median_norm"] = float(n_full.median())
            rec["max_norm"] = float(n_full.max())
            store[idx] = rec
        return post

    handles = [adapter.transformer.register_forward_pre_hook(
        lambda m, a, k: state.__setitem__("step", state["step"] + 1), with_kwargs=True)]
    for idx, name, b, kind in adapter.iter_blocks():
        if idx in layers:
            handles.append(b.register_forward_hook(make_post(idx), with_kwargs=True))
    try:
        for pid in prompt_ids:
            fp = out / f"maps_p{pid}_s{seed}.pt"
            if fp.exists():
                print("skip", fp.name); continue
            state["step"], store = -1, {}
            img = adapter.run(cfg.prompts[pid], seed)
            (out / "images").mkdir(exist_ok=True)
            img.save(out / "images" / f"p{pid}_s{seed}.png")
            torch.save({"prompt_id": pid, "seed": seed, "select": select,
                        "step": target_step, "maps": dict(store)}, fp)
            print("saved", fp.name)
    finally:
        for h in handles: h.remove()
        torch.cuda.empty_cache()
    return out


@torch.no_grad()
def capture_causal_maps(cfg: RunConfig, adapter, ablate_layer: int, observe_layer: int,
                        prompt_ids: Sequence[int] = (0, 1, 2),
                        seed: Optional[int] = None,
                        select: str = "global",
                        ks: Sequence[int] = (0, 1, 5, 10, 20),
                        fixed_channels: Optional[Sequence[int]] = None,
                        step: Optional[int] = None) -> Path:
    """CAUSAL mode. For each k: zero the top-k channels at `ablate_layer` (every
    step, all tokens) and record the norm map at `observe_layer`. k=0 = baseline.
    Channels are chosen from the CLEAN pass at ablate_layer, then held fixed, so
    the choice cannot drift as the intervention changes the activations."""
    seed = cfg.seeds[0] if seed is None else seed
    out = cfg.out / "spatial" / f"causal_L{ablate_layer}_obs{observe_layer}_{select}"
    out.mkdir(parents=True, exist_ok=True)
    target_step = (adapter.default_steps - 1) if step is None else step
    blocks = {idx: b for idx, n, b, k in adapter.iter_blocks()}

    for pid in prompt_ids:
        fp = out / f"maps_p{pid}_s{seed}.pt"
        if fp.exists():
            print("skip", fp.name); continue

        # pass 0: clean, to fix the channel ranking at the ablation layer
        picked, state = {}, {"step": -1}
        def pick(m, a, k, o):
            if state["step"] != target_step: return
            h = block_post_image(o)[0].float()
            for k_ in ks:
                if k_ > 0:
                    picked[k_] = _topk_channels(h, select, k_, fixed_channels).tolist()
        hs = [adapter.transformer.register_forward_pre_hook(
                  lambda m, a, k: state.__setitem__("step", state["step"] + 1), with_kwargs=True),
              blocks[ablate_layer].register_forward_hook(pick, with_kwargs=True)]
        img_clean = adapter.run(cfg.prompts[pid], seed)
        for h in hs: h.remove()

        maps = {}
        for k_ in ks:
            zero_ch = picked.get(k_, [])
            state2, rec = {"step": -1}, {}

            def zhook(m, a, kw):
                if not zero_ch: return None
                in_kw = isinstance(kw, dict) and "hidden_states" in kw
                x = kw["hidden_states"] if in_kw else (a[0] if a else None)
                if x is None: return None
                x = x.clone()
                for c in zero_ch:
                    x[0, :, c] = 0.0
                if in_kw:
                    kw = dict(kw); kw["hidden_states"] = x
                    return (a, kw)
                return ((x,) + tuple(a[1:]), kw)

            def obs(m, a, kw, o):
                if state2["step"] != target_step: return
                h = block_post_image(o)[0].float()
                n = h.norm(dim=-1)
                rec["norm"] = _grid(n, cfg)
                rec["x_top1"] = _grid(h[:, zero_ch[0]].abs(), cfg) if zero_ch else _grid(
                    h[:, _topk_channels(h, select, 1, fixed_channels)[0]].abs(), cfg)
                rec["median_norm"] = float(n.median()); rec["max_norm"] = float(n.max())
            hs = [adapter.transformer.register_forward_pre_hook(
                      lambda m, a, k: state2.__setitem__("step", state2["step"] + 1), with_kwargs=True),
                  blocks[ablate_layer].register_forward_pre_hook(zhook, with_kwargs=True),
                  blocks[observe_layer].register_forward_hook(obs, with_kwargs=True)]
            img = adapter.run(cfg.prompts[pid], seed)
            for h in hs: h.remove()
            (out / "images").mkdir(exist_ok=True)
            img.save(out / "images" / f"p{pid}_k{k_}.png")
            maps[k_] = rec | {"channels": zero_ch}
            torch.cuda.empty_cache()

        torch.save({"prompt_id": pid, "seed": seed, "select": select,
                    "ablate_layer": ablate_layer, "observe_layer": observe_layer,
                    "maps": maps}, fp)
        (out / "images").mkdir(exist_ok=True)
        img_clean.save(out / "images" / f"p{pid}_clean.png")
        print("saved", fp.name)
    return out


# ------------------------------------------------------------------- figures
def plot_subtraction_maps(cfg: RunConfig, select: str, layer: int,
                          prompt_ids: Sequence[int] = (0, 1, 2),
                          seed: Optional[int] = None,
                          ks: Sequence[int] = (1, 5, 10, 20),
                          shared_scale: bool = False) -> None:
    seed = cfg.seeds[0] if seed is None else seed
    d = cfg.out / "spatial" / f"subtract_{select}"
    caps = [torch.load(d / f"maps_p{p}_s{seed}.pt", weights_only=False) for p in prompt_ids]
    ncol = 3 + len(ks)
    fig, axes = plt.subplots(len(caps), ncol, figsize=(2.7 * ncol, 2.9 * len(caps)), squeeze=False)
    for r, cap in enumerate(caps):
        m = cap["maps"][layer]
        panels = [("generated", None), (f"isolated ch {m['top1_channel']}", m["isolated"]),
                  ("full norm", m["full"])] + [(f"minus top-{k}", m[f"minus{k}"]) for k in ks]
        norm_panels = [p[1] for p in panels[2:]]
        vmax = max(float(np.percentile(p, 99.5)) for p in norm_panels) if shared_scale else None
        for c, (title, arr) in enumerate(panels):
            ax = axes[r][c]
            if arr is None:
                ip = d / "images" / f"p{cap['prompt_id']}_s{seed}.png"
                if ip.exists(): ax.imshow(PILImage.open(ip))
            elif c == 1:
                ax.imshow(arr, cmap="inferno")
            else:
                ax.imshow(arr, cmap="viridis", vmin=0, vmax=vmax)
                ratio = arr.max() / max(np.median(arr), 1e-9)
                ax.set_xlabel(f"max/med {ratio:.1f}", fontsize=8)
            if r == 0: ax.set_title(title, fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        axes[r][0].set_ylabel(f"prompt {cap['prompt_id']}", fontsize=9)
    fig.suptitle(f"{cfg.model_key} SUBTRACTION (model untouched) — layer {layer}, "
                 f"channel select = {select}, shared colour scale = {shared_scale}", y=1.01)
    fig.tight_layout()
    fig.savefig(d / f"fig_subtract_L{layer}.png", dpi=160, bbox_inches="tight"); plt.show()


def plot_causal_maps(cfg: RunConfig, ablate_layer: int, observe_layer: int, select: str,
                     prompt_ids: Sequence[int] = (0, 1, 2), seed: Optional[int] = None,
                     shared_scale: bool = True) -> None:
    seed = cfg.seeds[0] if seed is None else seed
    d = cfg.out / "spatial" / f"causal_L{ablate_layer}_obs{observe_layer}_{select}"
    caps = [torch.load(d / f"maps_p{p}_s{seed}.pt", weights_only=False) for p in prompt_ids]
    ks = sorted(caps[0]["maps"].keys())
    ncol = 1 + len(ks)
    fig, axes = plt.subplots(len(caps), ncol, figsize=(2.7 * ncol, 2.9 * len(caps)), squeeze=False)
    for r, cap in enumerate(caps):
        arrs = [cap["maps"][k]["norm"] for k in ks]
        vmax = max(float(np.percentile(a, 99.5)) for a in arrs) if shared_scale else None
        ip = d / "images" / f"p{cap['prompt_id']}_clean.png"
        if ip.exists(): axes[r][0].imshow(PILImage.open(ip))
        axes[r][0].set_xticks([]); axes[r][0].set_yticks([])
        if r == 0: axes[r][0].set_title("generated", fontsize=9)
        for c, k in enumerate(ks, start=1):
            ax = axes[r][c]
            a = cap["maps"][k]["norm"]
            ax.imshow(a, cmap="viridis", vmin=0, vmax=vmax)
            ax.set_xlabel(f"max/med {a.max()/max(np.median(a),1e-9):.1f}", fontsize=8)
            if r == 0:
                ax.set_title("baseline" if k == 0 else f"zero top-{k}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        axes[r][0].set_ylabel(f"prompt {cap['prompt_id']}", fontsize=9)
    fig.suptitle(f"{cfg.model_key} CAUSAL: zero top-k channels at layer {ablate_layer}, "
                 f"observe norms at layer {observe_layer} (select = {select})", y=1.01)
    fig.tight_layout()
    fig.savefig(d / f"fig_causal_L{ablate_layer}_obs{observe_layer}.png", dpi=160,
                bbox_inches="tight"); plt.show()


def compare_selection_modes(cfg: RunConfig, layer: int, prompt_id: int = 0,
                            seed: Optional[int] = None) -> pd.DataFrame:
    """Which channel does each selection rule call 'top-1' at this layer?
    A mismatch here is the most likely source of cross-pipeline disagreement."""
    seed = cfg.seeds[0] if seed is None else seed
    rows = []
    for select in ("global", "loudest_token"):
        fp = cfg.out / "spatial" / f"subtract_{select}" / f"maps_p{prompt_id}_s{seed}.pt"
        if not fp.exists(): continue
        m = torch.load(fp, weights_only=False)["maps"].get(layer)
        if m is None: continue
        rows.append(dict(select=select, top1_channel=m["top1_channel"],
                         channels_top5=m["channels"].get(5),
                         ratio_full=round(m["max_norm"] / m["median_norm"], 2),
                         ratio_minus1=round(float(m["minus1"].max() / np.median(m["minus1"])), 2),
                         ratio_minus5=round(float(m["minus5"].max() / np.median(m["minus5"])), 2)))
    df = pd.DataFrame(rows)
    print(f"Selection-rule comparison at layer {layer}:")
    print(df.to_string(index=False))
    print("  Different top1_channel across rules -> the two pipelines are not removing "
          "the same thing, which alone can explain divergent conclusions.")
    return df


# ---- run ----
LAYERS = [10, 17, 18, 20, 24, 29, 36]          # include BOTH readings of "17"/"24"/"29"
for sel in ("global", "loudest_token"):
     capture_spatial_maps(cfg, adapter, LAYERS, select=sel)
     for L in (17, 24, 29):
         plot_subtraction_maps(cfg, sel, L)
compare_selection_modes(cfg, layer=17)
capture_causal_maps(cfg, adapter, ablate_layer=17, observe_layer=18, select="global")
plot_causal_maps(cfg, 17, 18, "global")
capture_causal_maps(cfg, adapter, ablate_layer=24, observe_layer=25, select="global")
plot_causal_maps(cfg, 24, 25, "global")

In [ ]:
# [CELL 12] ----------------------------------------------------------
# E8: channel removal, SUBTRACT vs ZERO, in the earlier figure layout
# -------------------------------------------------------------------------
# Columns:
#   generated | isolated channel C | isolated high-norm tokens
#   | high-norm tokens - channel C | - top 3 | - top 5 | - top 10 | - top 20
#
# Two methodologies over the SAME removal sets, so the figures stack:
#   SUBTRACT  bookkeeping only. Recompute the norm with those channels dropped.
#             The model never sees the change. (This is the earlier notebook's method.)
#   ZERO      causal. Set those channels to 0 in the residual stream at the
#             ablation layer (every denoising step), let the model run, and read
#             the norm map at the observation layer.
# observe_offset=0 measures both at the same layer, so the only difference
# between the two figures is that the model got to recompute.
#
# Design points:
#   * channel sets fixed from the CLEAN pass, so the intervention cannot change
#     which channels are removed
#   * exact MASKED norm removal (subtract-squares loses precision on ~pure tokens)
#   * ONE colour scale spanning BOTH figures per prompt
#   * every panel annotated with max/median and surviving register count
#   * top-k headers flag whether channel C is inside that set (nested vs rival)
#   * register tokens circled on the isolated-high-norm panel
#
# Requires cells 1-2 of the repro notebook (RunConfig, adapters, block_post_image).

from PIL import Image as PILImage

SELECT_MODES = ("global", "loudest_token", "fixed")


def _topk_channels(h: torch.Tensor, select: str, k: int,
                   fixed: Optional[Sequence[int]] = None) -> torch.Tensor:
    """h: [N, C] float. Indices of the k channels to remove.
       global        rank by token-mean |x|      (activation-landscape metric)
       loudest_token rank by |x| of the max-norm token
       fixed         a supplied list
    These rules disagree before the register is born, which alone can explain
    divergent conclusions between pipelines."""
    if select == "fixed":
        assert fixed is not None, "select='fixed' needs a channel list"
        return torch.tensor(list(fixed)[:k], dtype=torch.long)
    if select == "global":
        score = h.abs().mean(0)
    elif select == "loudest_token":
        score = h[h.norm(dim=-1).argmax()].abs()
    else:
        raise ValueError(select)
    return torch.topk(score, k).indices


def _norm_minus(h: torch.Tensor, ch: torch.Tensor) -> torch.Tensor:
    """Norm with the listed channels removed, by MASKING. The subtract-squares
    form suffers catastrophic cancellation exactly where it matters: when one
    channel holds ~99% of a token's energy, sum(x^2) - x_c^2 subtracts two
    nearly-equal large numbers. Masking is exact."""
    keep = torch.ones(h.shape[-1], dtype=torch.bool, device=h.device)
    keep[ch.to(h.device)] = False
    return h[:, keep].norm(dim=-1)


def _grid(v: torch.Tensor, cfg: RunConfig) -> np.ndarray:
    g = cfg.height // 16
    return v[: g * g].reshape(g, g).cpu().numpy()


@torch.no_grad()
def compare_selection_rules(cfg: RunConfig, adapter, layers: Sequence[int],
                            register_channel: int = 154, k: int = 5,
                            prompt_id: int = 0, seed: Optional[int] = None,
                            step: Optional[int] = None) -> pd.DataFrame:
    """One clean generation. Reports which channels each selection rule picks at
    each layer. Run this FIRST: if two pipelines use different rules they are not
    removing the same channels, and no layer-indexing argument can reconcile them."""
    seed = cfg.seeds[0] if seed is None else seed
    target_step = (adapter.default_steps - 1) if step is None else step
    rows, state = [], {"step": -1}

    def make_post(idx):
        def post(m, a, kk, o):
            if state["step"] != target_step or idx not in layers:
                return
            h = block_post_image(o)[0].float()
            n = h.norm(dim=-1); med = n.median()
            r = dict(layer=idx, ratio_full=round(float(n.max() / med), 2),
                     n_registers=int((n > 3.0 * med).sum()))
            for rule in ("global", "loudest_token"):
                ch = _topk_channels(h, rule, k)
                r[f"{rule}_top1"] = int(ch[0])
                r[f"{rule}_top{k}"] = ch.tolist()
                r[f"{rule}_has_ch{register_channel}"] = register_channel in ch.tolist()
            rows.append(r)
        return post

    handles = [adapter.transformer.register_forward_pre_hook(
        lambda m, a, kk: state.__setitem__("step", state["step"] + 1), with_kwargs=True)]
    for idx, name, b, kind in adapter.iter_blocks():
        if idx in layers:
            handles.append(b.register_forward_hook(make_post(idx), with_kwargs=True))
    try:
        adapter.run(cfg.prompts[prompt_id], seed)
    finally:
        for h in handles: h.remove()
        torch.cuda.empty_cache()

    df = pd.DataFrame(rows).sort_values("layer")
    print("Channel-selection rules per layer (clean pass):")
    print(df[["layer", "ratio_full", "n_registers", "global_top1", "loudest_token_top1",
              f"global_has_ch{register_channel}",
              f"loudest_token_has_ch{register_channel}"]].to_string(index=False))
    print("  Rules disagreeing on top1 -> the pipelines remove DIFFERENT channels.")
    return df


# ------------------------------------------------------------------------
# SUBTRACT vs ZERO, same removal sets, same layout as the earlier figure
# ------------------------------------------------------------------------
# Columns:  generated | isolated ch_reg | high-norm tokens (full norm)
#           | minus ch_reg | minus top-3 | minus top-5 | minus top-10 | minus top-20
#
# Two methodologies over the SAME removal sets:
#   SUBTRACT  bookkeeping only. Recompute the norm at the observation layer with
#             those channels dropped. The model never sees the change.
#   ZERO      causal. Set those channels to 0 in the residual stream at the
#             ablation layer (every denoising step), let the model run, and read
#             the norm map at the observation layer.
# With observe_offset=0 both are measured at the same layer, so the two rows are
# directly comparable and any difference is the model recomputing.
#
# Channel sets are fixed from the CLEAN pass, so the intervention cannot change
# which channels are being removed.

@torch.no_grad()
def capture_removal_comparison(cfg: RunConfig, adapter, layer: int,
                               register_channel: int = 154,
                               topks: Sequence[int] = (3, 5, 10, 20),
                               select: str = "global",
                               prompt_ids: Sequence[int] = (0, 1, 2, 3),
                               seed: Optional[int] = None,
                               step: Optional[int] = None,
                               observe_offset: int = 0) -> Path:
    seed = cfg.seeds[0] if seed is None else seed
    out = cfg.out / "spatial" / f"removal_L{layer}_off{observe_offset}_{select}"
    out.mkdir(parents=True, exist_ok=True)
    target_step = (adapter.default_steps - 1) if step is None else step
    blocks = {idx: b for idx, n, b, k in adapter.iter_blocks()}
    obs_layer = min(layer + observe_offset, max(blocks))

    for pid in prompt_ids:
        fp = out / f"removal_p{pid}_s{seed}.pt"
        if fp.exists():
            print("skip", fp.name); continue

        # ---------- pass 0: clean. fixes channel sets + all SUBTRACT maps ----------
        clean, state = {}, {"step": -1}

        def post_sel(m, a, k, o):                     # at the ABLATION layer: pick channels
            if state["step"] != target_step: return
            h = block_post_image(o)[0].float()
            sets = {f"ch{register_channel}": [register_channel]}
            for k_ in topks:
                sets[f"top{k_}"] = _topk_channels(h, select, k_).tolist()
            clean["sets"] = sets

        def post_obs(m, a, k, o):                     # at the OBSERVATION layer: subtract maps
            if state["step"] != target_step: return
            h = block_post_image(o)[0].float()
            n = h.norm(dim=-1); med = n.median()
            clean["isolated"] = _grid(h[:, register_channel].abs(), cfg)
            clean["full"] = _grid(n, cfg)
            clean["register_ids"] = torch.nonzero(n > 3.0 * med).flatten().tolist()
            clean["median_norm"] = float(med); clean["max_norm"] = float(n.max())
            clean["x_reg_max"] = float(h[:, register_channel].abs().max())
            clean["x_reg_median"] = float(h[:, register_channel].abs().median())
            clean["_h_obs"] = h                        # kept only inside this pass

        hs = [adapter.transformer.register_forward_pre_hook(
                  lambda m, a, k: state.__setitem__("step", state["step"] + 1), with_kwargs=True),
              blocks[layer].register_forward_hook(post_sel, with_kwargs=True)]
        if obs_layer != layer:
            hs.append(blocks[obs_layer].register_forward_hook(post_obs, with_kwargs=True))
        else:
            hs.append(blocks[layer].register_forward_hook(post_obs, with_kwargs=True))
        img_clean = adapter.run(cfg.prompts[pid], seed)
        for h_ in hs: h_.remove()

        h_obs = clean.pop("_h_obs")
        clean["subtract"] = {name: _grid(_norm_minus(h_obs, torch.tensor(ids)), cfg)
                             for name, ids in clean["sets"].items()}
        clean["subtract_stats"] = {}
        for name, ids in clean["sets"].items():
            v = _norm_minus(h_obs, torch.tensor(ids))
            clean["subtract_stats"][name] = dict(
                max=float(v.max()), median=float(v.median()),
                ratio=float(v.max() / v.median()),
                n_reg=int((v > 3.0 * v.median()).sum()))
        del h_obs
        torch.cuda.empty_cache()

        # ---------- one causal pass per removal set ----------
        zero_maps, zero_stats = {}, {}
        for name, ids in clean["sets"].items():
            st2, rec = {"step": -1}, {}

            def zhook(m, a, kw, _ids=ids):
                in_kw = isinstance(kw, dict) and "hidden_states" in kw
                x = kw["hidden_states"] if in_kw else (a[0] if a else None)
                if x is None: return None
                x = x.clone()
                for c in _ids:
                    x[0, :, c] = 0.0
                if in_kw:
                    kw = dict(kw); kw["hidden_states"] = x
                    return (a, kw)
                return ((x,) + tuple(a[1:]), kw)

            def obs(m, a, kw, o):
                if st2["step"] != target_step: return
                h = block_post_image(o)[0].float()
                n = h.norm(dim=-1); med = n.median()
                rec["map"] = _grid(n, cfg)
                rec["stats"] = dict(max=float(n.max()), median=float(med),
                                    ratio=float(n.max() / med),
                                    n_reg=int((n > 3.0 * med).sum()),
                                    x_reg_max=float(h[:, register_channel].abs().max()),
                                    x_reg_median=float(h[:, register_channel].abs().median()))

            hs = [adapter.transformer.register_forward_pre_hook(
                      lambda m, a, k: st2.__setitem__("step", st2["step"] + 1), with_kwargs=True),
                  blocks[layer].register_forward_pre_hook(zhook, with_kwargs=True),
                  blocks[obs_layer].register_forward_hook(obs, with_kwargs=True)]
            img = adapter.run(cfg.prompts[pid], seed)
            for h_ in hs: h_.remove()
            (out / "images").mkdir(exist_ok=True)
            img.save(out / "images" / f"p{pid}_zero_{name}.png")
            zero_maps[name] = rec["map"]; zero_stats[name] = rec["stats"]
            torch.cuda.empty_cache()

        (out / "images").mkdir(exist_ok=True)
        img_clean.save(out / "images" / f"p{pid}_clean.png")
        torch.save({"prompt_id": pid, "seed": seed, "layer": layer, "obs_layer": obs_layer,
                    "select": select, "register_channel": register_channel,
                    "topks": list(topks), "clean": clean,
                    "zero": zero_maps, "zero_stats": zero_stats}, fp)
        print("saved", fp.name)
    return out


def plot_removal_comparison(cfg: RunConfig, layer: int, register_channel: int = 154,
                            select: str = "global", prompt_ids: Sequence[int] = (0, 1, 2, 3),
                            seed: Optional[int] = None, observe_offset: int = 0,
                            shared_scale: bool = True, mark_registers: bool = True
                            ) -> pd.DataFrame:
    """One figure per methodology, identical layout, shared colour scale across BOTH
    so the two are directly comparable panel-for-panel."""
    seed = cfg.seeds[0] if seed is None else seed
    d = cfg.out / "spatial" / f"removal_L{layer}_off{observe_offset}_{select}"
    caps = [torch.load(d / f"removal_p{p}_s{seed}.pt", weights_only=False) for p in prompt_ids]
    reg = caps[0]["register_channel"]; topks = caps[0]["topks"]
    set_names = [f"ch{reg}"] + [f"top{k}" for k in topks]
    g = cfg.height // 16

    # one colour scale per prompt, spanning BOTH methodologies
    vmaxes = {}
    for cap in caps:
        arrs = [cap["clean"]["full"]] + [cap["clean"]["subtract"][s] for s in set_names] \
               + [cap["zero"][s] for s in set_names]
        vmaxes[cap["prompt_id"]] = max(float(np.percentile(a, 99.9)) for a in arrs)

    rows = []
    for mode in ("subtract", "zero"):
        # Column titles follow the earlier figure's naming scheme exactly.
        def _removal_label(s):
            what = f"channel {reg}" if s == f"ch{reg}" else f"top {s.replace('top', '')} channels"
            verb = "\u2212" if mode == "subtract" else "zeroed:"      # minus sign vs zeroed
            return (f"high-norm tokens\n{verb} {what}" if mode == "subtract"
                    else f"high-norm tokens\n{what} zeroed")
        cols = [("generated", "img"),
                (f"isolated channel {reg}", "iso"),
                ("isolated high-norm tokens", "full")] + \
               [(_removal_label(s), s) for s in set_names]
        fig, axes = plt.subplots(len(caps), len(cols),
                                 figsize=(2.8 * len(cols), 3.0 * len(caps)), squeeze=False)
        for r, cap in enumerate(caps):
            pid = cap["prompt_id"]; vmax = vmaxes[pid] if shared_scale else None
            for c, (title, key) in enumerate(cols):
                ax = axes[r][c]
                if key == "img":
                    ip = d / "images" / f"p{pid}_clean.png"
                    if ip.exists(): ax.imshow(PILImage.open(ip))
                elif key == "iso":
                    ax.imshow(cap["clean"]["isolated"], cmap="inferno")
                    ax.set_xlabel(f"max/med "
                                  f"{cap['clean']['x_reg_max']/max(cap['clean']['x_reg_median'],1e-9):.0f}",
                                  fontsize=8)
                elif key == "full":
                    arr = cap["clean"]["full"]
                    ax.imshow(arr, cmap="viridis", vmin=0, vmax=vmax)
                    ax.set_xlabel(f"max/med {cap['clean']['max_norm']/cap['clean']['median_norm']:.2f}",
                                  fontsize=8)
                    if mark_registers:
                        for t in cap["clean"]["register_ids"]:
                            ax.add_patch(plt.Circle((t % g, t // g), 1.4, fill=False,
                                                    color="red", lw=0.9))
                else:
                    arr = cap["clean"]["subtract"][key] if mode == "subtract" else cap["zero"][key]
                    st = (cap["clean"]["subtract_stats"][key] if mode == "subtract"
                          else cap["zero_stats"][key])
                    ax.imshow(arr, cmap="viridis", vmin=0, vmax=vmax)
                    ax.set_xlabel(f"max/med {st['ratio']:.2f} · {st['n_reg']} reg", fontsize=8)
                if r == 0:
                    extra = ""
                    if key in set_names and key != f"ch{reg}":
                        extra = ("\n(incl. ch %d)" % reg
                                 if reg in cap["clean"]["sets"][key] else "\n(excl. ch %d)" % reg)
                    ax.set_title(title + extra, fontsize=9)
                ax.set_xticks([]); ax.set_yticks([])
            axes[r][0].set_ylabel(f"prompt {pid}", fontsize=9)
        lbl = ("SUBTRACT — bookkeeping only, model untouched" if mode == "subtract"
               else f"ZERO — channels set to 0 at layer {layer}, model recomputes")
        fig.suptitle(f"{cfg.model_key} · {lbl} · observe layer {caps[0]['obs_layer']} · "
                     f"{cfg.height}x{cfg.width} · select={select}", y=1.005, fontsize=12)
        fig.tight_layout()
        fig.savefig(d / f"fig_{mode}_L{layer}.png", dpi=160, bbox_inches="tight")
        plt.show()

    for cap in caps:
        for s in set_names:
            rows.append(dict(prompt=cap["prompt_id"], removal=s,
                             in_topk_has_reg=(reg in cap["clean"]["sets"][s]),
                             ratio_clean=round(cap["clean"]["max_norm"] / cap["clean"]["median_norm"], 2),
                             ratio_subtract=round(cap["clean"]["subtract_stats"][s]["ratio"], 2),
                             ratio_zero=round(cap["zero_stats"][s]["ratio"], 2),
                             nreg_subtract=cap["clean"]["subtract_stats"][s]["n_reg"],
                             nreg_zero=cap["zero_stats"][s]["n_reg"],
                             x_reg_max_zero=round(cap["zero_stats"][s]["x_reg_max"], 1)))
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print(f"""
  ratio_subtract ~ 1  but  ratio_zero ~ ratio_clean
     -> the outliers ARE channel-{reg} spikes on paper, yet the model REBUILDS them
        when it is allowed to recompute. That gap is the maintenance loop.
  x_reg_max_zero close to the clean register level -> the channel was restored
     within the block, i.e. zeroing at this depth does not stick.""")
    return df


# ---- run (uncomment) ----------------------------------------------------
LAYERS = [17, 18, 24, 29, 36]

# 1. one generation: do the selection rules even agree?
compare_selection_rules(cfg, adapter, LAYERS, register_channel=REGISTER_CHANNEL)

# 2. per layer: 1 clean pass + 5 causal passes per prompt
for L in (17, 24, 36):
    capture_removal_comparison(cfg, adapter, layer=L,
                               register_channel=REGISTER_CHANNEL,
                               topks=(3, 5, 10, 20), prompt_ids=(0, 1, 2, 3))
    plot_removal_comparison(cfg, layer=L, register_channel=REGISTER_CHANNEL)

#optional: does zeroing stick further downstream?
capture_removal_comparison(cfg, adapter, layer=24, observe_offset=4,
                           register_channel=REGISTER_CHANNEL)
plot_removal_comparison(cfg, layer=24, observe_offset=4,
                        register_channel=REGISTER_CHANNEL)

In [ ]:
# [CELL 12] ----------------------------------------------------------------
# E8: channel removal, SUBTRACT vs ZERO, in the earlier figure layout
# -------------------------------------------------------------------------
# Columns:
#   generated | isolated channel C | isolated high-norm tokens
#   | high-norm tokens - channel C | - top 3 | - top 5 | - top 10 | - top 20
#
# Two methodologies over the SAME removal sets, so the figures stack:
#   SUBTRACT  bookkeeping only. Recompute the norm with those channels dropped.
#             The model never sees the change. (This is the earlier notebook's method.)
#   ZERO      causal. Set those channels to 0 in the residual stream at the
#             ablation layer (every denoising step), let the model run, and read
#             the norm map at the observation layer.
# observe_offset=0 measures both at the same layer, so the only difference
# between the two figures is that the model got to recompute.
#
# Design points:
#   * channel sets fixed from the CLEAN pass, so the intervention cannot change
#     which channels are removed
#   * exact MASKED norm removal (subtract-squares loses precision on ~pure tokens)
#   * ONE colour scale spanning BOTH figures per prompt
#   * every panel annotated with max/median and surviving register count
#   * top-k headers flag whether channel C is inside that set (nested vs rival)
#   * register tokens circled on the isolated-high-norm panel
#
# Requires cells 1-2 of the repro notebook (RunConfig, adapters, block_post_image).

from PIL import Image as PILImage

SELECT_MODES = ("global", "loudest_token", "fixed")


def _topk_channels(h: torch.Tensor, select: str, k: int,
                   fixed: Optional[Sequence[int]] = None) -> torch.Tensor:
    """h: [N, C] float. Indices of the k channels to remove.
       global        rank by token-mean |x|      (activation-landscape metric)
       loudest_token rank by |x| of the max-norm token
       fixed         a supplied list
    These rules disagree before the register is born, which alone can explain
    divergent conclusions between pipelines."""
    if select == "fixed":
        assert fixed is not None, "select='fixed' needs a channel list"
        return torch.tensor(list(fixed)[:k], dtype=torch.long)
    if select == "global":
        score = h.abs().mean(0)
    elif select == "loudest_token":
        score = h[h.norm(dim=-1).argmax()].abs()
    else:
        raise ValueError(select)
    return torch.topk(score, k).indices


def _norm_minus(h: torch.Tensor, ch: torch.Tensor) -> torch.Tensor:
    """Norm with the listed channels removed, by MASKING. The subtract-squares
    form suffers catastrophic cancellation exactly where it matters: when one
    channel holds ~99% of a token's energy, sum(x^2) - x_c^2 subtracts two
    nearly-equal large numbers. Masking is exact."""
    keep = torch.ones(h.shape[-1], dtype=torch.bool, device=h.device)
    keep[ch.to(h.device)] = False
    return h[:, keep].norm(dim=-1)


def _grid(v: torch.Tensor, cfg: RunConfig) -> np.ndarray:
    g = cfg.height // 16
    return v[: g * g].reshape(g, g).cpu().numpy()


@torch.no_grad()
def compare_selection_rules(cfg: RunConfig, adapter, layers: Sequence[int],
                            register_channel: int = 154, k: int = 5,
                            prompt_id: int = 0, seed: Optional[int] = None,
                            step: Optional[int] = None) -> pd.DataFrame:
    """One clean generation. Reports which channels each selection rule picks at
    each layer. Run this FIRST: if two pipelines use different rules they are not
    removing the same channels, and no layer-indexing argument can reconcile them."""
    seed = cfg.seeds[0] if seed is None else seed
    target_step = (adapter.default_steps - 1) if step is None else step
    rows, state = [], {"step": -1}

    def make_post(idx):
        def post(m, a, kk, o):
            if state["step"] != target_step or idx not in layers:
                return
            h = block_post_image(o)[0].float()
            n = h.norm(dim=-1); med = n.median()
            r = dict(layer=idx, ratio_full=round(float(n.max() / med), 2),
                     n_registers=int((n > 3.0 * med).sum()))
            for rule in ("global", "loudest_token"):
                ch = _topk_channels(h, rule, k)
                r[f"{rule}_top1"] = int(ch[0])
                r[f"{rule}_top{k}"] = ch.tolist()
                r[f"{rule}_has_ch{register_channel}"] = register_channel in ch.tolist()
            rows.append(r)
        return post

    handles = [adapter.transformer.register_forward_pre_hook(
        lambda m, a, kk: state.__setitem__("step", state["step"] + 1), with_kwargs=True)]
    for idx, name, b, kind in adapter.iter_blocks():
        if idx in layers:
            handles.append(b.register_forward_hook(make_post(idx), with_kwargs=True))
    try:
        adapter.run(cfg.prompts[prompt_id], seed)
    finally:
        for h in handles: h.remove()
        torch.cuda.empty_cache()

    df = pd.DataFrame(rows).sort_values("layer")
    print("Channel-selection rules per layer (clean pass):")
    print(df[["layer", "ratio_full", "n_registers", "global_top1", "loudest_token_top1",
              f"global_has_ch{register_channel}",
              f"loudest_token_has_ch{register_channel}"]].to_string(index=False))
    print("  Rules disagreeing on top1 -> the pipelines remove DIFFERENT channels.")
    return df


# ------------------------------------------------------------------------
# SUBTRACT vs ZERO, same removal sets, same layout as the earlier figure
# ------------------------------------------------------------------------
# Columns:  generated | isolated ch_reg | high-norm tokens (full norm)
#           | minus ch_reg | minus top-3 | minus top-5 | minus top-10 | minus top-20
#
# Two methodologies over the SAME removal sets:
#   SUBTRACT  bookkeeping only. Recompute the norm at the observation layer with
#             those channels dropped. The model never sees the change.
#   ZERO      causal. Set those channels to 0 in the residual stream at the
#             ablation layer (every denoising step), let the model run, and read
#             the norm map at the observation layer.
# With observe_offset=0 both are measured at the same layer, so the two rows are
# directly comparable and any difference is the model recomputing.
#
# Channel sets are fixed from the CLEAN pass, so the intervention cannot change
# which channels are being removed.

@torch.no_grad()
def capture_removal_comparison(cfg: RunConfig, adapter, layer: int,
                               register_channel: int = 154,
                               topks: Sequence[int] = (3, 5, 10, 20),
                               select: str = "global",
                               prompt_ids: Sequence[int] = (0, 1, 2, 3),
                               seed: Optional[int] = None,
                               step: Optional[int] = None,
                               observe_offset: int = 0,
                               zero_steps: Optional[Sequence[int]] = None) -> Path:
    seed = cfg.seeds[0] if seed is None else seed
    ztag = "all" if zero_steps is None else "-".join(map(str, sorted(zero_steps)))
    out = cfg.out / "spatial" / f"removal_L{layer}_off{observe_offset}_z{ztag}_{select}"
    out.mkdir(parents=True, exist_ok=True)
    target_step = (adapter.default_steps - 1) if step is None else step
    # zero_steps=None zeroes at EVERY denoising step, so the observed step also
    # carries compounded trajectory drift from all earlier steps. Passing
    # [target_step] isolates the block's IMMEDIATE response with no compounding.
    zsteps = (set(range(adapter.default_steps)) if zero_steps is None else set(zero_steps))
    blocks = {idx: b for idx, n, b, k in adapter.iter_blocks()}
    obs_layer = min(layer + observe_offset, max(blocks))

    for pid in prompt_ids:
        fp = out / f"removal_p{pid}_s{seed}.pt"
        if fp.exists():
            print("skip", fp.name); continue

        # ---------- pass 0: clean. fixes channel sets + all SUBTRACT maps ----------
        clean, state = {}, {"step": -1}

        def post_sel(m, a, k, o):                     # at the ABLATION layer: pick channels
            if state["step"] != target_step: return
            h = block_post_image(o)[0].float()
            sets = {f"ch{register_channel}": [register_channel]}
            for k_ in topks:
                sets[f"top{k_}"] = _topk_channels(h, select, k_).tolist()
            clean["sets"] = sets

        def post_obs(m, a, k, o):                     # at the OBSERVATION layer: subtract maps
            if state["step"] != target_step: return
            h = block_post_image(o)[0].float()
            n = h.norm(dim=-1); med = n.median()
            clean["isolated"] = _grid(h[:, register_channel].abs(), cfg)
            clean["full"] = _grid(n, cfg)
            clean["register_ids"] = torch.nonzero(n > 3.0 * med).flatten().tolist()
            clean["median_norm"] = float(med); clean["max_norm"] = float(n.max())
            clean["x_reg_max"] = float(h[:, register_channel].abs().max())
            clean["x_reg_median"] = float(h[:, register_channel].abs().median())
            clean["argmax_token"] = int(n.argmax())
            clean["_h_obs"] = h                        # kept only inside this pass

        hs = [adapter.transformer.register_forward_pre_hook(
                  lambda m, a, k: state.__setitem__("step", state["step"] + 1), with_kwargs=True),
              blocks[layer].register_forward_hook(post_sel, with_kwargs=True)]
        if obs_layer != layer:
            hs.append(blocks[obs_layer].register_forward_hook(post_obs, with_kwargs=True))
        else:
            hs.append(blocks[layer].register_forward_hook(post_obs, with_kwargs=True))
        img_clean = adapter.run(cfg.prompts[pid], seed)
        for h_ in hs: h_.remove()

        h_obs = clean.pop("_h_obs")
        clean["subtract"] = {name: _grid(_norm_minus(h_obs, torch.tensor(ids)), cfg)
                             for name, ids in clean["sets"].items()}
        clean["subtract_stats"] = {}
        for name, ids in clean["sets"].items():
            v = _norm_minus(h_obs, torch.tensor(ids))
            clean["subtract_stats"][name] = dict(
                max=float(v.max()), median=float(v.median()),
                ratio=float(v.max() / v.median()),
                n_reg=int((v > 3.0 * v.median()).sum()))
        del h_obs
        torch.cuda.empty_cache()

        # ---------- one causal pass per removal set ----------
        zero_maps, zero_stats = {}, {}
        for name, ids in clean["sets"].items():
            st2, rec = {"step": -1}, {}

            def zhook(m, a, kw, _ids=ids):
                if st2["step"] not in zsteps:
                    return None
                in_kw = isinstance(kw, dict) and "hidden_states" in kw
                x = kw["hidden_states"] if in_kw else (a[0] if a else None)
                if x is None: return None
                x = x.clone()
                for c in _ids:
                    x[0, :, c] = 0.0
                if in_kw:
                    kw = dict(kw); kw["hidden_states"] = x
                    return (a, kw)
                return ((x,) + tuple(a[1:]), kw)

            def obs(m, a, kw, o):
                if st2["step"] != target_step: return
                h = block_post_image(o)[0].float()
                n = h.norm(dim=-1); med = n.median()
                rec["map"] = _grid(n, cfg)
                cr = clean["register_ids"]
                rec["stats"] = dict(
                    max=float(n.max()), median=float(med),
                    ratio=float(n.max() / med),
                    n_reg=int((n > 3.0 * med).sum()),
                    # max over ALL tokens: cannot tell restoration from replacement
                    x_reg_max=float(h[:, register_channel].abs().max()),
                    x_reg_median=float(h[:, register_channel].abs().median()),
                    # channel value on the tokens that WERE registers in the clean run:
                    # this is restoration; a large x_reg_max with a small value here
                    # means a DIFFERENT token took over.
                    x_reg_on_clean_regs=(float(h[cr, register_channel].abs().mean())
                                         if cr else float("nan")),
                    argmax_token=int(n.argmax()),
                    same_max_token=bool(int(n.argmax()) == clean["argmax_token"]))

            hs = [adapter.transformer.register_forward_pre_hook(
                      lambda m, a, k: st2.__setitem__("step", st2["step"] + 1), with_kwargs=True),
                  blocks[layer].register_forward_pre_hook(zhook, with_kwargs=True),
                  blocks[obs_layer].register_forward_hook(obs, with_kwargs=True)]
            img = adapter.run(cfg.prompts[pid], seed)
            for h_ in hs: h_.remove()
            (out / "images").mkdir(exist_ok=True)
            img.save(out / "images" / f"p{pid}_zero_{name}.png")
            zero_maps[name] = rec["map"]; zero_stats[name] = rec["stats"]
            torch.cuda.empty_cache()

        (out / "images").mkdir(exist_ok=True)
        img_clean.save(out / "images" / f"p{pid}_clean.png")
        torch.save({"prompt_id": pid, "seed": seed, "layer": layer, "obs_layer": obs_layer,
                    "select": select, "register_channel": register_channel,
                    "zero_steps": sorted(zsteps), "topks": list(topks), "clean": clean,
                    "zero": zero_maps, "zero_stats": zero_stats}, fp)
        print("saved", fp.name)
    return out


def plot_removal_comparison(cfg: RunConfig, layer: int, register_channel: int = 154,
                            select: str = "global", prompt_ids: Sequence[int] = (0, 1, 2, 3),
                            seed: Optional[int] = None, observe_offset: int = 0,
                            zero_steps: Optional[Sequence[int]] = None,
                            seeds: Optional[Sequence[int]] = None,
                            shared_scale: bool = True, mark_registers: bool = True
                            ) -> pd.DataFrame:
    """Two figures (SUBTRACT, ZERO), identical layout, shared colour scale across
    BOTH so they are comparable panel-for-panel. Figures use the first seed;
    the returned table pools every seed in `seeds`."""
    seed = cfg.seeds[0] if seed is None else seed
    seeds = [seed] if seeds is None else list(seeds)
    ztag = "all" if zero_steps is None else "-".join(map(str, sorted(zero_steps)))
    d = cfg.out / "spatial" / f"removal_L{layer}_off{observe_offset}_z{ztag}_{select}"
    caps = [torch.load(d / f"removal_p{p}_s{seeds[0]}.pt", weights_only=False) for p in prompt_ids]
    reg = caps[0]["register_channel"]; topks = caps[0]["topks"]
    set_names = [f"ch{reg}"] + [f"top{k}" for k in topks]
    g = cfg.height // 16

    # one colour scale per prompt, spanning BOTH methodologies
    vmaxes = {}
    for cap in caps:
        arrs = [cap["clean"]["full"]] + [cap["clean"]["subtract"][s] for s in set_names] \
               + [cap["zero"][s] for s in set_names]
        vmaxes[cap["prompt_id"]] = max(float(np.percentile(a, 99.9)) for a in arrs)

    rows = []
    for mode in ("subtract", "zero"):
        # Column titles follow the earlier figure's naming scheme exactly.
        def _removal_label(s):
            what = f"channel {reg}" if s == f"ch{reg}" else f"top {s.replace('top', '')} channels"
            verb = "\u2212" if mode == "subtract" else "zeroed:"      # minus sign vs zeroed
            return (f"high-norm tokens\n{verb} {what}" if mode == "subtract"
                    else f"high-norm tokens\n{what} zeroed")
        cols = [("generated", "img"),
                (f"isolated channel {reg}", "iso"),
                ("isolated high-norm tokens", "full")] + \
               [(_removal_label(s), s) for s in set_names]
        fig, axes = plt.subplots(len(caps), len(cols),
                                 figsize=(2.8 * len(cols), 3.0 * len(caps)), squeeze=False)
        for r, cap in enumerate(caps):
            pid = cap["prompt_id"]; vmax = vmaxes[pid] if shared_scale else None
            for c, (title, key) in enumerate(cols):
                ax = axes[r][c]
                if key == "img":
                    ip = d / "images" / f"p{pid}_clean.png"
                    if ip.exists(): ax.imshow(PILImage.open(ip))
                elif key == "iso":
                    ax.imshow(cap["clean"]["isolated"], cmap="inferno")
                    ax.set_xlabel(f"max/med "
                                  f"{cap['clean']['x_reg_max']/max(cap['clean']['x_reg_median'],1e-9):.0f}",
                                  fontsize=8)
                elif key == "full":
                    arr = cap["clean"]["full"]
                    ax.imshow(arr, cmap="viridis", vmin=0, vmax=vmax)
                    ax.set_xlabel(f"max/med {cap['clean']['max_norm']/cap['clean']['median_norm']:.2f}",
                                  fontsize=8)
                    if mark_registers:
                        for t in cap["clean"]["register_ids"]:
                            ax.add_patch(plt.Circle((t % g, t // g), 1.4, fill=False,
                                                    color="red", lw=0.9))
                else:
                    arr = cap["clean"]["subtract"][key] if mode == "subtract" else cap["zero"][key]
                    st = (cap["clean"]["subtract_stats"][key] if mode == "subtract"
                          else cap["zero_stats"][key])
                    ax.imshow(arr, cmap="viridis", vmin=0, vmax=vmax)
                    ax.set_xlabel(f"max/med {st['ratio']:.2f} · {st['n_reg']} reg", fontsize=8)
                if r == 0:
                    extra = ""
                    if key in set_names and key != f"ch{reg}":
                        extra = ("\n(incl. ch %d)" % reg
                                 if reg in cap["clean"]["sets"][key] else "\n(excl. ch %d)" % reg)
                    ax.set_title(title + extra, fontsize=9)
                ax.set_xticks([]); ax.set_yticks([])
            axes[r][0].set_ylabel(f"prompt {pid}", fontsize=9)
        lbl = ("SUBTRACT — bookkeeping only, model untouched" if mode == "subtract"
               else f"ZERO — channels set to 0 at layer {layer}, model recomputes")
        fig.suptitle(f"{cfg.model_key} · {lbl} · observe layer {caps[0]['obs_layer']} · "
                     f"{cfg.height}x{cfg.width} · select={select}", y=1.005, fontsize=12)
        fig.tight_layout()
        fig.savefig(d / f"fig_{mode}_L{layer}.png", dpi=160, bbox_inches="tight")
        plt.show()

    all_caps = []
    for sd in seeds:
        for p in prompt_ids:
            fp = d / f"removal_p{p}_s{sd}.pt"
            if fp.exists():
                all_caps.append(torch.load(fp, weights_only=False))
    for cap in all_caps:
        for s in set_names:
            rows.append(dict(prompt=cap["prompt_id"], seed=cap["seed"], removal=s,
                             in_topk_has_reg=(reg in cap["clean"]["sets"][s]),
                             ratio_clean=round(cap["clean"]["max_norm"] / cap["clean"]["median_norm"], 2),
                             ratio_subtract=round(cap["clean"]["subtract_stats"][s]["ratio"], 2),
                             ratio_zero=round(cap["zero_stats"][s]["ratio"], 2),
                             nreg_subtract=cap["clean"]["subtract_stats"][s]["n_reg"],
                             nreg_zero=cap["zero_stats"][s]["n_reg"],
                             x_reg_max_zero=round(cap["zero_stats"][s]["x_reg_max"], 1),
                             x_reg_on_clean_regs=round(
                                 cap["zero_stats"][s].get("x_reg_on_clean_regs", float("nan")), 1),
                             same_max_token=cap["zero_stats"][s].get("same_max_token")))
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    agg = (df.groupby("removal")[["ratio_clean", "ratio_subtract", "ratio_zero",
                                  "x_reg_max_zero", "x_reg_on_clean_regs"]]
           .agg(["mean", "std"]).round(2))
    print("\nPooled over prompts and seeds:")
    print(agg.to_string())
    zs = caps[0].get("zero_steps")
    print(f"""
  zero_steps = {zs}  ({'EVERY step: the observed step also carries compounded'
                       ' trajectory drift' if len(zs) > 1 else
                       'single step: isolates the block immediate response'})

  How to read the two zero columns:
    x_reg_on_clean_regs ~ clean register level  -> the ORIGINAL registers were
        restored (true maintenance / homeostasis)
    x_reg_max_zero >> x_reg_on_clean_regs       -> a DIFFERENT token took over
        (replacement, not restoration); check same_max_token
    both >> clean level                          -> genuine OVERSHOOT, only
        believable if it survives zero_steps=[final] and replicates across seeds
  nreg_subtract = 0 while nreg_zero > 0 -> registers are channel-{reg} spikes on
    paper, but the model regenerates them when allowed to recompute.""")
    return df


# ---- run (uncomment) ----------------------------------------------------
LAYERS = [17, 18, 24, 29, 36]
SEEDS  = [0, 1, 2]

# 1. one generation: do the selection rules agree on which channel is top-1?
compare_selection_rules(cfg, adapter, LAYERS, register_channel=REGISTER_CHANNEL)

# 2. CLEAN causal measurement: zero only at the observed step, 3 seeds.
#    This removes the multi-step trajectory confound.
for sd in SEEDS:
    capture_removal_comparison(cfg, adapter, layer=24, seed=sd,
                               register_channel=REGISTER_CHANNEL,
                               topks=(3, 5, 10, 20), prompt_ids=(0, 1, 2, 3),
                               zero_steps=[adapter.default_steps - 1])
plot_removal_comparison(cfg, layer=24, register_channel=REGISTER_CHANNEL,
                        zero_steps=[adapter.default_steps - 1], seeds=SEEDS)

# 3. birth test: zero BEFORE birth, observe AFTER it
for sd in SEEDS:
    capture_removal_comparison(cfg, adapter, layer=17, observe_offset=1, seed=sd,
                               register_channel=REGISTER_CHANNEL,
                               zero_steps=[adapter.default_steps - 1])
plot_removal_comparison(cfg, layer=17, observe_offset=1,
                        register_channel=REGISTER_CHANNEL,
                        zero_steps=[adapter.default_steps - 1], seeds=SEEDS)

# 4. optional: the every-step version, for comparison with the single-step run
capture_removal_comparison(cfg, adapter, layer=24, register_channel=REGISTER_CHANNEL)
plot_removal_comparison(cfg, layer=24, register_channel=REGISTER_CHANNEL)

In [ ]:
# [CELL 13] -----------------------------------------------------------
# E9: the register control law, measure the block's write as a function of
#      the register channel's incoming value.
# -------------------------------------------------------------------------
# Motivation. Zeroing the register channel at a mid-zone block makes that block
# write ~40k to the channel in a single block, against a clean-state write of
# order +-50. That is the signature of negative feedback. If the loop is
# (approximately) linear we should be able to write it down:
#
#       write  ~  g * (setpoint - x_in)
#
# Method. At the observed denoising step only (no trajectory compounding),
# SCALE the register channel at the block input by alpha, then read the same
# tokens at the block output. Because every earlier step is clean, the input is
# exactly alpha * x_in_clean, so
#
#       write(alpha) = x_out(alpha) - alpha * x_in_clean
#
# Regressing write on the actual input value gives the loop gain (slope) and the
# setpoint (x-intercept). R^2 says whether "controller" is the right model at all.
#
# scope="all"        scale the channel in EVERY token (matches the zeroing runs;
#                    also perturbs the floor and the LayerNorm statistics)
# scope="registers"  scale it only in the clean register tokens (isolates the
#                    block's response to the REGISTER's own value)
# Running both separates "responds to this token" from "responds to the channel
# statistics across tokens".
#
# For FLUX single blocks the write is additionally split into attention vs MLP
# by slicing row `register_channel` of proj_out.weight, which is exact because
# proj_out consumes cat[attn_out, mlp_hidden].
#
# Requires cells 1-2 of the repro notebook.

from dataclasses import dataclass


@dataclass
class ControlLawConfig:
    layer: int = 24
    register_channel: int = 154
    alphas: Tuple[float, ...] = (0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 4.0)
    scope: str = "all"                      # "all" | "registers"
    prompt_ids: Tuple[int, ...] = (0, 1, 2)
    seeds: Tuple[int, ...] = (0, 1)
    spike_ratio: float = 3.0
    decompose: bool = True                  # attention vs MLP split (single blocks)


class ChannelScaleHook:
    """Scale one channel at a block's input, on the chosen tokens, at chosen steps."""

    def __init__(self, channel: int, alpha: float, steps: set,
                 token_ids: Optional[Sequence[int]] = None):
        self.channel, self.alpha, self.steps = channel, alpha, steps
        self.token_ids = None if token_ids is None else torch.tensor(list(token_ids))
        self.step = -1
        self.applied_input: Optional[torch.Tensor] = None   # x_in AFTER scaling, for audit

    def __call__(self, module, args, kwargs):
        if self.step not in self.steps:
            return None
        in_kw = isinstance(kwargs, dict) and "hidden_states" in kwargs
        x = kwargs["hidden_states"] if in_kw else (args[0] if args else None)
        if x is None:
            return None
        x = x.clone()
        if self.token_ids is None:
            x[0, :, self.channel] = x[0, :, self.channel] * self.alpha
        else:
            idx = self.token_ids.to(x.device)
            x[0, idx, self.channel] = x[0, idx, self.channel] * self.alpha
        self.applied_input = x[0, :, self.channel].detach().float().cpu()
        if in_kw:
            kwargs = dict(kwargs); kwargs["hidden_states"] = x
            return (args, kwargs)
        return ((x,) + tuple(args[1:]), kwargs)


@torch.no_grad()
def run_control_law(cfg: RunConfig, adapter, cl: ControlLawConfig,
                    step: Optional[int] = None) -> pd.DataFrame:
    out = cfg.out / "control_law" / f"L{cl.layer}_{cl.scope}"
    out.mkdir(parents=True, exist_ok=True)
    target_step = (adapter.default_steps - 1) if step is None else step
    blocks = {idx: b for idx, n, b, k in adapter.iter_blocks()}
    kinds = {idx: k for idx, n, b, k in adapter.iter_blocks()}
    block = blocks[cl.layer]
    C = cl.register_channel

    for pid in cl.prompt_ids:
        for sd in cl.seeds:
            fp = out / f"cl_p{pid}_s{sd}.csv"
            if fp.exists():
                print("skip", fp.name); continue

            # ---------- clean reference: x_in and x_out on this block ----------
            ref, st = {}, {"step": -1}

            def pre_clean(m, a, kw):
                if st["step"] != target_step: return None
                x = kw.get("hidden_states", a[0] if a else None)
                ref["x_in"] = x[0, :, C].detach().float().cpu()
                return None

            def post_clean(m, a, kw, o):
                if st["step"] != target_step: return
                h = block_post_image(o)[0].float()
                n = h.norm(dim=-1)
                ref["x_out"] = h[:, C].detach().cpu()
                ref["reg_ids"] = torch.nonzero(n > cl.spike_ratio * n.median()).flatten().tolist()
                ref["median_norm"] = float(n.median())

            hs = [adapter.transformer.register_forward_pre_hook(
                      lambda m, a, kw: st.__setitem__("step", st["step"] + 1), with_kwargs=True),
                  block.register_forward_pre_hook(pre_clean, with_kwargs=True),
                  block.register_forward_hook(post_clean, with_kwargs=True)]
            adapter.run(cfg.prompts[pid], sd)
            for h_ in hs: h_.remove()
            reg_ids = ref["reg_ids"]
            if not reg_ids:
                print(f"  p{pid} s{sd}: no registers at layer {cl.layer}, skipping")
                continue

            rows = []
            # clean point (alpha = 1, no hook) for the audit trail
            rows.append(dict(prompt_id=pid, seed=sd, alpha=1.0, hooked=False,
                             x_in=float(ref["x_in"][reg_ids].mean()),
                             x_out=float(ref["x_out"][reg_ids].mean()),
                             write=float((ref["x_out"] - ref["x_in"])[reg_ids].mean()),
                             n_reg=len(reg_ids)))

            # ---------- alpha sweep ----------
            for alpha in cl.alphas:
                hook = ChannelScaleHook(C, alpha, {target_step},
                                        reg_ids if cl.scope == "registers" else None)
                st2, rec = {"step": -1}, {}
                attrib = {"attn": None, "mlp": None}

                def tf_pre(m, a, kw):
                    st2["step"] += 1; hook.step = st2["step"]
                    return None

                def post(m, a, kw, o):
                    if st2["step"] != target_step: return
                    h = block_post_image(o)[0].float()
                    rec["x_out"] = h[:, C].detach().cpu()

                hs = [adapter.transformer.register_forward_pre_hook(tf_pre, with_kwargs=True),
                      block.register_forward_pre_hook(hook, with_kwargs=True),
                      block.register_forward_hook(post, with_kwargs=True)]

                # exact attention/MLP split of the channel write (FLUX single blocks)
                if cl.decompose and kinds[cl.layer] == "single" and hasattr(block, "proj_out"):
                    D = adapter.transformer.config.num_attention_heads * \
                        adapter.transformer.config.attention_head_dim
                    w = block.proj_out.weight[C].detach()

                    def proj_pre(m, a, kw=None):
                        if st2["step"] != target_step: return None
                        catv = a[0] if a else None
                        if catv is None: return None
                        v = catv[0].float()
                        img0 = v.shape[0] - len(ref["x_in"])          # image tokens are the tail
                        sel = v[img0 + torch.tensor(reg_ids)]
                        attrib["attn"] = float((sel[:, :D] @ w[:D].float()).mean())
                        attrib["mlp"] = float((sel[:, D:] @ w[D:].float()).mean())
                        return None
                    hs.append(block.proj_out.register_forward_pre_hook(proj_pre))

                adapter.run(cfg.prompts[pid], sd)
                for h_ in hs: h_.remove()

                x_in_eff = (hook.applied_input if hook.applied_input is not None
                            else ref["x_in"])
                xi = float(x_in_eff[reg_ids].mean())
                xo = float(rec["x_out"][reg_ids].mean())
                rows.append(dict(prompt_id=pid, seed=sd, alpha=alpha, hooked=True,
                                 x_in=xi, x_out=xo, write=xo - xi, n_reg=len(reg_ids),
                                 attn_write=attrib["attn"], mlp_write=attrib["mlp"]))
                torch.cuda.empty_cache()

            pd.DataFrame(rows).to_csv(fp, index=False)
            print("saved", fp.name)
    return pd.concat([pd.read_csv(f) for f in sorted(out.glob("cl_*.csv"))],
                     ignore_index=True)


def analyze_control_law(cfg: RunConfig, cl: ControlLawConfig) -> dict:
    out = cfg.out / "control_law" / f"L{cl.layer}_{cl.scope}"
    df = pd.concat([pd.read_csv(f) for f in sorted(out.glob("cl_*.csv"))], ignore_index=True)
    d = df[df["hooked"]].copy()
    if d.empty:
        print("no data"); return {}

    x, y = d["x_in"].to_numpy(), d["write"].to_numpy()
    A = np.vstack([x, np.ones_like(x)]).T
    (slope, intercept), *_ = np.linalg.lstsq(A, y, rcond=None)
    pred = A @ np.array([slope, intercept])
    ss_res = float(((y - pred) ** 2).sum())
    ss_tot = float(((y - y.mean()) ** 2).sum())
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    setpoint = -intercept / slope if slope != 0 else float("nan")
    clean_x = float(df[~df["hooked"]]["x_out"].mean())

    print(f"E9 CONTROL LAW — layer {cl.layer}, channel {cl.register_channel}, scope={cl.scope}")
    print(f"  fit   write = {slope:.3f} * x_in + {intercept:,.0f}     R^2 = {r2:.3f}   (n={len(d)})")
    print(f"  loop gain            g = {-slope:.3f}")
    print(f"  implied setpoint       = {setpoint:,.0f}")
    print(f"  clean register level   = {clean_x:,.0f}   (setpoint/clean = {setpoint/clean_x:.2f})")
    print("""
  slope < 0 with high R^2 -> proportional negative feedback; the block writes to
     drive the channel back toward the setpoint, and |slope| is the loop gain.
  setpoint close to the clean register level -> the controller's target IS the
     natural register magnitude (self-consistent homeostat).
  slope ~ 0 -> the block writes a fixed amount regardless of input: not a
     controller but a constant source; the overshoot would then be trivial.
  |slope| > 1 -> overcorrection on a step input (overshoot), consistent with
     what the zeroing runs showed.""")

    per_alpha = d.groupby("alpha").agg(
        x_in=("x_in", "mean"), x_out=("x_out", "mean"), write=("write", "mean"),
        write_sd=("write", "std")).round(0)
    print("\nPer-alpha means:")
    print(per_alpha.to_string())

    if "mlp_write" in d.columns and d["mlp_write"].notna().any():
        dd = d.dropna(subset=["mlp_write"])
        share = (dd["mlp_write"].abs() /
                 (dd["mlp_write"].abs() + dd["attn_write"].abs() + 1e-9)).mean()
        print(f"\n  pathway split of the write: MLP share = {share:.2f} "
              f"(attn mean {dd['attn_write'].mean():,.0f}, mlp mean {dd['mlp_write'].mean():,.0f})")
        print("  MLP share ~1 -> the correction travels the MLP path, matching the "
              "clean-state maintenance attribution.")

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
    ax[0].scatter(x, y, s=22, alpha=0.7)
    xs = np.linspace(0, max(x.max(), setpoint * 1.1), 100)
    ax[0].plot(xs, slope * xs + intercept, "k--", lw=1.2,
               label=f"write = {slope:.2f}·x + {intercept:,.0f}  (R²={r2:.2f})")
    ax[0].axhline(0, color="gray", lw=0.7)
    ax[0].axvline(setpoint, color="tab:red", ls=":", lw=1.2, label=f"setpoint {setpoint:,.0f}")
    ax[0].axvline(clean_x, color="tab:green", ls=":", lw=1.2, label=f"clean level {clean_x:,.0f}")
    ax[0].set_xlabel(f"channel {cl.register_channel} value entering the block")
    ax[0].set_ylabel("write by the block")
    ax[0].set_title(f"{cfg.model_key} L{cl.layer}: register control law ({cl.scope})")
    ax[0].legend(fontsize=8)

    ax[1].errorbar(per_alpha.index, per_alpha["x_out"], yerr=None, marker="o")
    ax[1].axhline(clean_x, color="tab:green", ls=":", label="clean level")
    ax[1].set_xlabel("alpha (input scaling)"); ax[1].set_ylabel("channel value LEAVING the block")
    ax[1].set_title("Output vs input scaling\n(flat = perfect regulation; rising = partial)")
    ax[1].legend(fontsize=8)
    fig.tight_layout(); fig.savefig(out / "fig_control_law.png", dpi=180); plt.show()

    res = dict(slope=float(slope), gain=float(-slope), intercept=float(intercept),
               r2=float(r2), setpoint=float(setpoint), clean_level=clean_x, n=int(len(d)))
    json.dump(res, open(out / "control_law_fit.json", "w"), indent=2)
    return res


# ---- run (uncomment) ----------------------------------------------------
cl = ControlLawConfig(layer=24, register_channel=REGISTER_CHANNEL, scope="all")
run_control_law(cfg, adapter, cl)
analyze_control_law(cfg, cl)

# isolate the register's own value from the channel-wide statistics
cl_reg = ControlLawConfig(layer=24, register_channel=REGISTER_CHANNEL, scope="registers")
run_control_law(cfg, adapter, cl_reg)
analyze_control_law(cfg, cl_reg)

# depth profile of the gain: is the controller strongest mid-zone?
for L in (18, 20, 24, 28, 32, 36):
    c = ControlLawConfig(layer=L, register_channel=REGISTER_CHANNEL, scope="all",
                         prompt_ids=(0, 1), seeds=(0,))
    run_control_law(cfg, adapter, c); analyze_control_law(cfg, c)